# Supplementary SageMath notebook for Section 3.3

This notebook contains the verification code and the explicit nonlinear
subspace path for the construction described in Section 3.3 of the paper.

The experiment is motivated by the Poseidon Bounty Program setting. It should
be understood as a verification of the internal round nonlinear subspace
construction, rather than as a full attack on the complete challenge instance.

We consider a Poseidon-type instance over the KoalaBear field

\[
p=2^{31}-2^{24}+1,
\]

with state size

\[
t=16.
\]

The motivating constraint pattern is the CICO-2 setting: two input coordinates
and two output coordinates are fixed. Therefore, in the internal round
subspace construction, the number of available extra constraints is

\[
Ec = t-4 = 12.
\]

The purpose of this notebook is to construct and verify a nonlinear subspace
path for the internal rounds under this constraint budget. The computation
follows the method described in Section 3.3: it first constructs the required
nonlinear subspace constraints, then verifies that the resulting path satisfies
the desired degeneration and consistency conditions.

## Environment

The notebook was written and tested with **SageMath 9.5** using the SageMath
Jupyter kernel. It is not a plain Python notebook.

To run the notebook, start Jupyter from SageMath, for example by

```bash
sage -n jupyter
```

and then open this `.ipynb` file with the SageMath kernel.

## Organization of the notebook

The notebook is divided into four parts.

### Part 1: Helper routines

The first part contains auxiliary functions for finite-field computations,
polynomial substitutions, Poseidon state updates, Macaulay matrix construction,
rank tests, and related sanity checks.

This part is largely shared with the supplementary notebook for Section 3.1.
Some comments and diagnostic messages in this part are non-English because
these routines were originally developed during exploratory computations. This
does not affect execution. Reviewers who only want to reproduce the
verification can simply run these cells in order.

### Part 2: Construction of the nonlinear subspace path

The second part implements the construction method used in Section 3.3. It
uses the available constraint budget \(E_c=12\) to construct a nonlinear
subspace path for the internal rounds of the \(t=16\) KoalaBear instance.

This part is the main constructive step of the notebook. It determines the
constraints that define the nonlinear path.

### Part 3: Verification of the constructed ideal

The third part verifies that the constructed ideal satisfies
the expected degeneration properties.

More precisely, it checks the relevant ideal-membership, reduction, and
dimension conditions predicted by the nonlinear subspace construction in
Section 3.3. This provides a direct computational verification that the
constructed ideal has the intended behavior along the internal rounds.

### Part 4: Saving the explicit polynomial path

The fourth part records the explicit polynomial path to external files. 
These saved outputs are included to make the construction
inspectable and to provide the explicit data referred to in the paper.

The main reproducible computation starts from **Part 2**. **Part 1** only
provides supporting routines required by the later cells.

### Part 1: Helper routines

The first part contains auxiliary functions for finite-field computations,
polynomial substitutions, Poseidon state updates, Macaulay matrix construction,
rank tests, and related sanity checks.

This part is largely shared with the supplementary notebook for Section 3.1.
Some comments and diagnostic messages in this part are non-English because
these routines were originally developed during exploratory computations. This
does not affect execution. Reviewers who only want to reproduce the
verification can simply run these cells in order.

In [1]:
def convert_to_grelex(eqs):
    """
    把多项式列表转换到 grelex (deglex) 排序下
    返回 (new_eqs, new_ring)
    """
    R_old = eqs[0].parent()
    F = R_old.base_ring()
    var_names = R_old.variable_names()

    # 新建 deglex 排序的多项式环
    R_new = PolynomialRing(F, var_names, order='deglex')

    # 构造变量替换映射
    subs_dict = {R_old.gen(i): R_new.gen(i) for i in range(R_old.ngens())}

    new_eqs = [R_new(f.subs(subs_dict)) for f in eqs]

    return new_eqs, R_new

# def monomials_by_degree(eqs):
#     """
#     返回:
#     { degree : set( monomials ) }
#     """
#     R = eqs[0].parent()
#     result = {}
#     for f in eqs:
#         for mon, coeff in f.dict().items():
#             # mon 是指数tuple
#             deg = sum(mon)

#             if deg not in result:
#                 result[deg] = set()

#             result[deg].add(mon)
#     return result

def monomials_by_degree(eqs):
    """
    返回:
    { degree : set( exponent_tuples ) }
    exponent_tuples 全部是 Python tuple
    """
    result = {}
    for f in eqs:
        for mon, coeff in f.dict().items():
            mon = tuple(mon)          # ✅ 强制统一类型
            deg = sum(mon)
            result.setdefault(deg, set()).add(mon)
    return result

from itertools import combinations_with_replacement

def all_monomials_of_degree(k, d):
    """
    返回所有 degree=d 的指数tuple
    """
    from sage.all import IntegerVectors
    return [tuple(v) for v in IntegerVectors(d, k)]

def analyze_monomial_support(eqs):
    """
    输出每个degree：
        出现了哪些单项式
        缺少哪些单项式
    """
    R = eqs[0].parent()
    k = R.ngens()

    support = monomials_by_degree(eqs)

    max_deg = max(support.keys())

    for d in sorted(support.keys()):
        print("\n========================")
        print(f"Degree {d}")
        print("========================")

        present = support[d]
        all_possible = set(all_monomials_of_degree(k, d))
        missing = all_possible - present

        print(f"Total possible: {len(all_possible)}")
        print(f"Present: {len(present)}")
        print(f"Missing: {len(missing)}")

        print("\nPresent monomials (exponent vectors):")
        for m in sorted(present):
            print(m)

        print("\nMissing monomials:")
        for m in sorted(missing):
            print(m)

def make_ring(p, nvars=4):
    S = PolynomialRing(GF(p), 't')
    # t = vector(S, S.gen())
    t = S.gen()

    R = PolynomialRing(S, nvars, 'x', order='degrevlex')
    X = vector(R, R.gens())

    return R, X, t

R, X, t = make_ring(1009, 4)
print(R)
print(X)
print(t)

from sage.combinat.integer_vector import IntegerVectors

def monomials_of_degree(R, X, d):
    mons = []
    n = len(X)

    for exps in IntegerVectors(d, n):
        m = R(1)
        for xi, ei in zip(X, exps):
            m *= xi**ei
        mons.append(m)

    return mons
print(monomials_of_degree(R, X, 2))

def monomials_leq_degree(R, X, D):
    mons = []

    for d in range(D+1):
        mons += monomials_of_degree(R, X, d)

    return mons
mons = monomials_leq_degree(R, X, 2)
print(mons)

Multivariate Polynomial Ring in x0, x1, x2, x3 over Univariate Polynomial Ring in t over Finite Field of size 1009
(x0, x1, x2, x3)
t
[x0^2, x0*x1, x0*x2, x0*x3, x1^2, x1*x2, x1*x3, x2^2, x2*x3, x3^2]
[1, x0, x1, x2, x3, x0^2, x0*x1, x0*x2, x0*x3, x1^2, x1*x2, x1*x3, x2^2, x2*x3, x3^2]


In [2]:
def span_equations_from_compressed_matrix(Mcrit):
    """
    对压缩后的矩阵 Mcrit：
    - 前 n-1 行是线性无关前缀基
    - 最后一行是目标行
    构造“最后一行属于前缀行张成空间”的方程组。
    """

    R = Mcrit.base_ring()
    K = R.base_ring()

    r = Mcrit.nrows()
    m = r - 1
    ncols = Mcrit.ncols()

    lam_names = [f"lam{i}" for i in range(m)]
    S = PolynomialRing(K, list(R.variable_names()) + lam_names, order='lex')

    # 嵌入到新环
    sub = {R(v): S(v) for v in R.variable_names()}

    prefix = []
    for i in range(m):
        prefix.append([S(Mcrit[i,j].subs(sub)) for j in range(ncols)])

    last = [S(Mcrit[m,j].subs(sub)) for j in range(ncols)]
    lam = [S(name) for name in lam_names]

    eqs_span = []
    for j in range(ncols):
        expr = last[j] - sum(lam[i] * prefix[i][j] for i in range(m))
        if expr != 0:
            eqs_span.append(expr)

    return S, eqs_span, lam


def random_search_span_system(S, eqs_span, fixed_subs, free_vars, trials=20000, p=101, seed=1234):
    set_random_seed(seed)

    sols = []
    free_vars_S = [S(str(v)) for v in free_vars]

    for _ in range(trials):
        subs = dict(fixed_subs)
        for v in free_vars_S:
            subs[v] = GF(p).random_element()

        ok = True
        for f in eqs_span:
            if f.subs(subs) != 0:
                ok = False
                break

        if ok:
            sols.append(subs)
            print("found solution =", subs)
            break

    return sols


def run_random_slice_univariate_support_only(M, keep_var='t0', trials=100, seed=1234):
    if seed is not None:
        set_random_seed(seed)

    R0 = M.base_ring()
    base = R0.base_ring()

    # 只收集 M 中真正出现的原变量
    used_var_set = set()
    for i in range(M.nrows()):
        for j in range(M.ncols()):
            f = M[i, j]
            if f != 0:
                used_var_set.update(f.variables())

    used_vars = sorted(used_var_set, key=str)

    print("M.nrows() =", M.nrows())
    print("M.ncols() =", M.ncols())
    print("used original variables =", used_vars)
    print("number of used original variables =", len(used_vars))

    # 新环：只包含真正出现的原变量 + 新 c 变量
    c_names = [f'c{i}' for i in range(M.nrows())]
    var_names = [str(v) for v in used_vars] + c_names
    R = PolynomialRing(base, var_names, order='degrevlex')

    # 构造从旧环到新环的替换映射
    sub_to_R = {v: R(str(v)) for v in used_vars}

    # 把 M 搬到新环，但只按出现变量替换
    M_new = matrix(
        R,
        M.nrows(),
        M.ncols(),
        [
            R(M[i, j].subs(sub_to_R)) if M[i, j] != 0 else R.zero()
            for i in range(M.nrows())
            for j in range(M.ncols())
        ]
    )

    vars_all = list(R.gens())
    c_vars = vars_all[len(used_vars):]

    keep = R(keep_var)
    if keep not in vars_all:
        raise ValueError(f"keep_var={keep_var} 不在实际使用变量里。可选变量: {[str(v) for v in vars_all]}")

    # 构造 c*M 的列方程
    eqs = []
    for j in range(M_new.ncols()):
        eq = sum(c_vars[i] * M_new[i, j] for i in range(M_new.nrows()))
        if eq != 0:
            eqs.append(eq)

    print("方程数量:", len(eqs))
    print("总变量数(实际使用变量 + c):", len(vars_all))
    print("变量列表:", vars_all)
    print("保留变量:", keep)
    print("=" * 50)

    def randval():
        if hasattr(base, "is_finite") and base.is_finite():
            L = list(base)
            return L[ZZ.random_element(len(L))]
        return ZZ.random_element(-5, 6)

    for tr in range(trials):
        # 固定 c0 = 1，排除平凡零解
        subs = {c_vars[0]: base(1)}

        for v in vars_all:
            if v != keep and v != c_vars[0]:
                subs[v] = randval()

        sliced = [f.subs(subs) for f in eqs]
        sliced = [f for f in sliced if f != 0]

        if len(sliced) == 0:
            print(f"[trial {tr}] 全部为 0 -> {keep} 自由")
            continue

        # 检查是否真成了单变量
        if any(any(w != keep for w in f.variables()) for f in sliced):
            print(f"[trial {tr}] 代入后不是单变量，跳过")
            continue

        S = PolynomialRing(base, str(keep))
        polys = [S(f) for f in sliced if S(f) != 0]

        if len(polys) == 0:
            print(f"[trial {tr}] 转单变量环后全 0 -> {keep} 自由")
            continue

        g = polys[0]
        for f in polys[1:]:
            g = gcd(g, f)

        print(f"[trial {tr}] gcd = {g}")
        if g.degree() <= 0:
            print("   -> 无解")
        else:
            try:
                roots = g.roots(multiplicities=False)
            except Exception:
                roots = "无法直接求"
            print("   -> 有解, roots =", roots)

def reduce_macaulay_columns_symbolic(M, verbose=True, return_data=False):
    """
    对多项式矩阵 M 的列做“符号线性去冗余”。

    方法：
      - 收集所有条目中出现过的单项式
      - 把每一列展开成一个普通系数向量
      - 在底层系数域上做列消元
      - 保留线性无关列

    输入：
      M : entries 在多项式环中的矩阵
      verbose : 是否打印信息
      return_data : 是否额外返回中间数据

    输出：
      M_reduced
      keep_cols
      （若 return_data=True，还返回 coeff_mat, monomials）
    """

    R = M.base_ring()
    K = R.base_ring()

    nrows = M.nrows()
    ncols = M.ncols()

    if verbose:
        print("========================================")
        print("原矩阵大小:", nrows, "x", ncols)
        print("base ring =", R)
        print("coefficient field/ring =", K)

    # 1. 收集所有出现过的单项式
    monomial_set = set()
    for i in range(nrows):
        for j in range(ncols):
            f = M[i, j]
            if f != 0:
                # f.monomials() 返回出现过的单项式
                monomial_set.update(f.monomials())

    monomials = sorted(monomial_set, key=str)
    nm = len(monomials)
    mon_index = {m: k for k, m in enumerate(monomials)}

    if verbose:
        print("出现过的不同单项式数:", nm)

    # 2. 构造普通系数矩阵：
    #    行索引 = (原矩阵行号, 单项式)
    #    列索引 = 原矩阵列号
    #
    #    shape = (nrows * nm) x ncols
    coeff_mat = Matrix(K, nrows * nm, ncols)

    for j in range(ncols):
        for i in range(nrows):
            f = M[i, j]
            if f == 0:
                continue
            d = f.dict()   # exponent tuple -> coefficient
            for exp, coeff in d.items():
                mono = R.monomial(*exp)
                row_idx = i * nm + mon_index[mono]
                coeff_mat[row_idx, j] = coeff

    if verbose:
        print("展开后的系数矩阵大小:", coeff_mat.nrows(), "x", coeff_mat.ncols())

    # 3. 找列主元（保留线性无关列）
    #
    # Sage 的 echelon_form/transformation 对列不总是最方便，
    # 所以转置后找行主元：coeff_mat 的列独立 <=> coeff_mat^T 的行独立
    E = coeff_mat.transpose().echelon_form()
    pivot_rows = E.pivots()

    # 但上面这个 pivot_rows 不是原列索引，需要改用右核来更稳？
    # 更直接的办法：对 coeff_mat 做列空间基提取
    #
    # Sage 里更稳妥：逐列增量加入，看 rank 是否上升

    keep_cols = []
    current = Matrix(K, coeff_mat.nrows(), 0)

    current_rank = 0
    for j in range(ncols):
        test = current.augment(coeff_mat.column(j))
        r = test.rank()
        if r > current_rank:
            keep_cols.append(j)
            current = test
            current_rank = r

    M_reduced = M.matrix_from_columns(keep_cols)

    if verbose:
        print("保留列数:", len(keep_cols))
        print("删除列数:", ncols - len(keep_cols))
        print("保留列索引:", keep_cols)
        print("缩减后矩阵大小:", M_reduced.nrows(), "x", M_reduced.ncols())
        print("========================================")

    if return_data:
        return M_reduced, keep_cols, coeff_mat, monomials
    else:
        return M_reduced, keep_cols

In [3]:
import random
from itertools import product

# =========================================================
# 基础工具
# =========================================================

def _normalize_point_dict_for_ring(pt, ring_or_gens):
    """
    统一成 { "t0": val0, "t1": val1, ... } 形式
    支持:
      - dict: {"t0":..., "t1":...}
      - list/tuple: [..]  (按变量顺序)
    """
    gens = ring_or_gens.gens() if hasattr(ring_or_gens, "gens") else tuple(ring_or_gens)

    if isinstance(pt, dict):
        out = {}
        for g in gens:
            name = str(g)
            if name in pt:
                out[name] = pt[name]
        return out

    if isinstance(pt, (list, tuple)):
        if len(pt) != len(gens):
            raise ValueError(f"需要 {len(gens)} 个参数值，但给了 {len(pt)} 个")
        return {str(gens[i]): pt[i] for i in range(len(gens))}

    raise TypeError("pt 必须是 dict / list / tuple")


def _point_dict_to_subs(M, pt):
    """
    把点 dict/list 变成适合 substitute 的 {t_i: val} 字典
    """
    S = M.base_ring()
    gens = list(S.gens())
    pt_dict = _normalize_point_dict_for_ring(pt, gens)
    F = S.base_ring()

    subs = {}
    for g in gens:
        name = str(g)
        if name not in pt_dict:
            raise ValueError(f"点里缺少变量 {name}")
        subs[g] = F(pt_dict[name])
    return subs


def specialize_matrix_over_base_field(M, pt):
    """
    把关于 t 的矩阵 M 代值为底域上的数值矩阵
    """
    S = M.base_ring()
    F = S.base_ring()
    subs = _point_dict_to_subs(M, pt)
    return matrix(F, M.substitute(subs))


def check_last_row_in_span_numeric(M, pt, verbose=False):
    """
    数值检查：最后一行是否在前面行张成空间里
    """
    M0 = specialize_matrix_over_base_field(M, pt)
    r_full = M0.rank()
    r_pref = M0[:-1].rank()
    ok = (r_full == r_pref)

    if verbose:
        print("point =", _normalize_point_dict_for_ring(pt, M.base_ring()))
        print("rank(full)   =", r_full)
        print("rank(prefix) =", r_pref)
        print("last row in span? ", ok)

    return {
        "matrix": M0,
        "rank_full": r_full,
        "rank_prefix": r_pref,
        "ok": ok,
    }


def _random_point_for_matrix(M, fixed_assignments=None, free_vars=None, value_pool=None):
    """
    生成一个随机点:
      - fixed_assignments: dict, 固定某些 t
      - free_vars: 只随机这些变量；若为 None 则随机所有未固定变量
      - value_pool: 可迭代对象；若 None 则从底域随机取
    """
    S = M.base_ring()
    F = S.base_ring()
    gens = list(S.gens())
    fixed_assignments = {} if fixed_assignments is None else dict(fixed_assignments)

    fixed_assignments = {str(k): F(v) for k, v in fixed_assignments.items()}

    if free_vars is None:
        free_names = [str(t) for t in gens if str(t) not in fixed_assignments]
    else:
        free_names = [str(t) for t in free_vars if str(t) not in fixed_assignments]

    pt = {}
    for g in gens:
        name = str(g)
        if name in fixed_assignments:
            pt[name] = F(fixed_assignments[name])
        elif name in free_names:
            if value_pool is None:
                pt[name] = F.random_element()
            else:
                vals = list(value_pool)
                pt[name] = F(random.choice(vals))
        else:
            # 不在 fixed，也不在 free_vars 里的，默认随机
            if value_pool is None:
                pt[name] = F.random_element()
            else:
                vals = list(value_pool)
                pt[name] = F(random.choice(vals))

    return pt


def _enumerate_points_for_matrix(M, fixed_assignments=None, free_vars=None, value_pool=None, max_points=None):
    """
    枚举点：
      - 如果 free_vars 给出，且 value_pool 可迭代，则枚举它们的笛卡尔积
      - 其余变量按 fixed_assignments 固定；没有指定的变量默认填 0
    用于低维切片小范围扫描
    """
    S = M.base_ring()
    F = S.base_ring()
    gens = list(S.gens())
    fixed_assignments = {} if fixed_assignments is None else dict(fixed_assignments)
    fixed_assignments = {str(k): F(v) for k, v in fixed_assignments.items()}

    if free_vars is None:
        raise ValueError("枚举模式下请显式给出 free_vars")

    if value_pool is None:
        raise ValueError("枚举模式下请给出 value_pool")

    free_names = [str(t) for t in free_vars]
    vals = [F(v) for v in value_pool]

    count = 0
    for tup in product(vals, repeat=len(free_names)):
        pt = {}
        for g in gens:
            name = str(g)
            if name in fixed_assignments:
                pt[name] = fixed_assignments[name]
            elif name in free_names:
                idx = free_names.index(name)
                pt[name] = tup[idx]
            else:
                pt[name] = F.zero()

        yield pt
        count += 1
        if max_points is not None and count >= max_points:
            return


# =========================================================
# 子式复杂度与选取
# =========================================================

def entry_t_degree(a):
    """
    a 是系数环 S=F[t...] 里的元素，返回 t-总次数
    零元记为 -1
    """
    if a == 0:
        return -1
    try:
        return a.total_degree()
    except Exception:
        try:
            mons = a.monomials()
            if len(mons) == 0:
                return 0
            return max(sum(m.degree(v) for v in a.parent().gens()) for m in mons)
        except Exception:
            return 0


def column_complexity_score(M, j, row_indices=None):
    """
    一列的复杂度打分：次数越低越好，非零越少越好
    """
    if row_indices is None:
        row_indices = range(M.nrows())

    deg_sum = 0
    nnz = 0
    for i in row_indices:
        a = M[i, j]
        if a != 0:
            nnz += 1
            d = entry_t_degree(a)
            deg_sum += max(d, 0)

    # 先看次数，再看非零数
    return (deg_sum, nnz, j)


def minor_complexity_score(M, col_set):
    """
    一个方阵子式的粗略复杂度：条目次数和 + 非零数
    """
    deg_sum = 0
    nnz = 0
    n = len(col_set)
    row_indices = list(range(n))
    for i in row_indices:
        for j in col_set:
            a = M[i, j]
            if a != 0:
                nnz += 1
                d = entry_t_degree(a)
                deg_sum += max(d, 0)
    return (deg_sum, nnz, tuple(col_set))


def find_prefix_pivot_columns_numeric(M, trials=30, value_pool=None, fixed_assignments=None, seed=None, verbose=True):
    """
    在随机特化点上找 prefix(前 nrows-1 行) 的一个 15 列 pivot chart
    返回长度为 nrows-1 的列集 J，使 prefix[:,J] 常常可逆
    """
    if seed is not None:
        random.seed(seed)

    nrows = M.nrows()
    ncols = M.ncols()
    pref_rows = nrows - 1

    if pref_rows <= 0:
        raise ValueError("矩阵行数至少要 >= 2")

    S = M.base_ring()
    F = S.base_ring()

    for trial in range(trials):
        pt = _random_point_for_matrix(
            M,
            fixed_assignments=fixed_assignments,
            free_vars=None,
            value_pool=value_pool
        )
        M0 = specialize_matrix_over_base_field(M, pt)
        P0 = M0[:pref_rows, :]

        r = P0.rank()
        if r < pref_rows:
            continue

        try:
            piv = list(P0.pivots())
        except Exception:
            E = P0.echelon_form()
            piv = list(E.pivots())

        if len(piv) >= pref_rows:
            J = piv[:pref_rows]
            if verbose:
                print(f"[find_prefix_pivot_columns_numeric] found at trial {trial}")
                print("pivot columns J =", J)
            return J

    raise RuntimeError("没有在随机点上找到满秩 prefix 的 pivot 列集；可增加 trials 或换 fixed_assignments")


def build_test_minors_from_pivot_chart(M, pivot_cols=None, num_test=6, candidate_pool=20, seed=None, verbose=True):
    """
    构造若干个 16x16 测试子式：
      - 行固定为全部 16 行
      - 列为 pivot_cols(15列) + 1个额外列
    这样比乱抽 16x16 更贴近“最后一行是否落入 prefix span”
    """
    if seed is not None:
        random.seed(seed)

    nrows = M.nrows()
    ncols = M.ncols()
    pref_rows = nrows - 1

    if pivot_cols is None:
        pivot_cols = find_prefix_pivot_columns_numeric(M, verbose=verbose)

    pivot_cols = list(pivot_cols)
    if len(pivot_cols) != pref_rows:
        raise ValueError(f"pivot_cols 长度应为 {pref_rows}，现在是 {len(pivot_cols)}")

    rest = [j for j in range(ncols) if j not in pivot_cols]

    # 优先挑复杂度较低的额外列
    scored = []
    for j in rest:
        # 主要关注全部16行上的复杂度；也可改成只看最后一行+少量行
        score = column_complexity_score(M, j, row_indices=range(nrows))
        scored.append(score)

    scored.sort()
    extras = [x[2] for x in scored[:candidate_pool]]

    # 再从候选里取前 num_test 个
    test_minors = []
    for k in extras[:num_test]:
        cols = list(pivot_cols) + [k]
        test_minors.append(cols)

    if verbose:
        print("[build_test_minors_from_pivot_chart]")
        print("pivot_cols =", pivot_cols)
        print("chosen extra columns =", [cols[-1] for cols in test_minors])
        print("num test minors =", len(test_minors))
        for idx, cols in enumerate(test_minors):
            print(f"  minor {idx}: extra={cols[-1]}, score={minor_complexity_score(M, cols)[:2]}")

    return {
        "pivot_cols": pivot_cols,
        "test_minors": test_minors,
    }


# =========================================================
# 子式 det 检查
# =========================================================

def minor_det_numeric(M, col_set, pt):
    """
    数值点上计算全部行、指定列的方阵 det
    """
    M0 = specialize_matrix_over_base_field(M, pt)
    A = M0.matrix_from_columns(col_set)
    return A.det()


def evaluate_test_minors_numeric(M, test_minors, pt, stop_at_first_nonzero=True):
    """
    在点 pt 上检查一组测试子式 det
    返回:
      {
        "all_zero": bool,
        "det_values": [...],
        "first_bad_index": int / None
      }
    """
    det_values = []
    first_bad = None

    for idx, cols in enumerate(test_minors):
        d = minor_det_numeric(M, cols, pt)
        det_values.append(d)
        if d != 0 and first_bad is None:
            first_bad = idx
            if stop_at_first_nonzero:
                return {
                    "all_zero": False,
                    "det_values": det_values,
                    "first_bad_index": first_bad,
                }

    return {
        "all_zero": (first_bad is None),
        "det_values": det_values,
        "first_bad_index": first_bad,
    }


# =========================================================
# 主搜索函数
# =========================================================

def search_points_by_minor_filter(
    M,
    test_minors=None,
    pivot_cols=None,
    num_test=6,
    candidate_pool=20,
    fixed_assignments=None,
    free_vars=None,
    value_pool=None,
    mode="random",          # "random" / "enumerate"
    max_trials=2000,
    stop_after_first_good=True,
    seed=None,
    verbose=True,
):
    """
    主函数：
      1) 若未给 test_minors，则自动构造
      2) 先检查测试子式 det 是否全 0
      3) 通过后再检查 rank(full)==rank(prefix)

    返回 summary 字典
    """
    if seed is not None:
        random.seed(seed)

    if test_minors is None:
        built = build_test_minors_from_pivot_chart(
            M,
            pivot_cols=pivot_cols,
            num_test=num_test,
            candidate_pool=candidate_pool,
            seed=seed,
            verbose=verbose
        )
        pivot_cols = built["pivot_cols"]
        test_minors = built["test_minors"]
    else:
        pivot_cols = pivot_cols

    good_points = []
    minor_pass_points = []

    stats = {
        "tested_points": 0,
        "minor_pass": 0,
        "rank_pass": 0,
        "first_good_point": None,
    }

    if mode == "enumerate":
        point_iter = _enumerate_points_for_matrix(
            M,
            fixed_assignments=fixed_assignments,
            free_vars=free_vars,
            value_pool=value_pool,
            max_points=max_trials
        )
    elif mode == "random":
        def _rand_iter():
            for _ in range(max_trials):
                yield _random_point_for_matrix(
                    M,
                    fixed_assignments=fixed_assignments,
                    free_vars=free_vars,
                    value_pool=value_pool
                )
        point_iter = _rand_iter()
    else:
        raise ValueError("mode 只能是 'random' 或 'enumerate'")

    for it, pt in enumerate(point_iter):
        stats["tested_points"] += 1

        minor_info = evaluate_test_minors_numeric(
            M,
            test_minors,
            pt,
            stop_at_first_nonzero=True
        )

        if not minor_info["all_zero"]:
            if verbose and (it + 1) % 100 == 0:
                print(f"[trial {it+1}] minor filter not passed")
            continue

        stats["minor_pass"] += 1
        minor_pass_points.append(dict(pt))

        rank_info = check_last_row_in_span_numeric(M, pt, verbose=False)

        if rank_info["ok"]:
            stats["rank_pass"] += 1
            good_points.append({
                "point": dict(pt),
                "rank_full": rank_info["rank_full"],
                "rank_prefix": rank_info["rank_prefix"],
                "minor_values": minor_info["det_values"],
            })

            if stats["first_good_point"] is None:
                stats["first_good_point"] = dict(pt)

            if verbose:
                print(f"[trial {it+1}] FOUND GOOD POINT")
                print("point =", pt)
                print("rank(full) =", rank_info["rank_full"])
                print("rank(prefix) =", rank_info["rank_prefix"])
                print("minor dets =", minor_info["det_values"])

            if stop_after_first_good:
                break
        else:
            if verbose:
                print(f"[trial {it+1}] minors all zero, but rank check failed")
                print("point =", pt)
                print("rank(full) =", rank_info["rank_full"])
                print("rank(prefix) =", rank_info["rank_prefix"])

    summary = {
        "pivot_cols": pivot_cols,
        "test_minors": test_minors,
        "stats": stats,
        "minor_pass_points": minor_pass_points,
        "good_points": good_points,
        "first_good_point": stats["first_good_point"],
    }

    if verbose:
        print("\n========== search summary ==========")
        print("tested_points =", stats["tested_points"])
        print("minor_pass    =", stats["minor_pass"])
        print("rank_pass     =", stats["rank_pass"])
        print("first_good_point =", stats["first_good_point"])

    return summary


import random
import time

def _normalize_fixed_assignments_for_matrix(M, fixed_assignments=None):
    """
    fixed_assignments 可写成:
      {"t0": 1, "t1": 2}
    或
      {T[0]: 1, T[1]: 2}
    统一转成 {str(var): field_element}
    """
    S = M.base_ring()
    F = S.base_ring()
    fixed_assignments = {} if fixed_assignments is None else dict(fixed_assignments)

    out = {}
    for k, v in fixed_assignments.items():
        out[str(k)] = F(v)
    return out


def random_point_for_matrix(M, fixed_assignments=None, free_vars=None, value_pool=None):
    """
    生成一个随机点:
      - fixed_assignments: 固定部分变量
      - free_vars: 若给出，则只把这些变量视为“要随机选的重点变量”；
                  其他未固定变量也会随机，只是写法上保留这个接口方便以后改
      - value_pool: 例如 range(50)，则从这些值里随机取
    """
    S = M.base_ring()
    F = S.base_ring()
    gens = list(S.gens())

    fixed = _normalize_fixed_assignments_for_matrix(M, fixed_assignments)

    if free_vars is None:
        free_names = set(str(g) for g in gens if str(g) not in fixed)
    else:
        free_names = set(str(x) for x in free_vars if str(x) not in fixed)

    pt = {}
    for g in gens:
        name = str(g)
        if name in fixed:
            pt[name] = fixed[name]
        else:
            if value_pool is None:
                pt[name] = F.random_element()
            else:
                vals = list(value_pool)
                pt[name] = F(random.choice(vals))
    return pt


def specialize_matrix_numeric(M, pt):
    """
    把符号矩阵 M 代入点 pt，得到底域上的数值矩阵
    """
    S = M.base_ring()
    F = S.base_ring()
    gens = list(S.gens())

    subs = {}
    for g in gens:
        name = str(g)
        if name not in pt:
            raise ValueError(f"点里缺少变量 {name}")
        subs[g] = F(pt[name])

    return matrix(F, M.substitute(subs))


def rank_gap_at_point(M, pt, prefix_rows=None, return_matrix=False):
    """
    默认检查“最后一行是否被前面所有行 span 到”
    即比较:
      prefix = M0[:-1, :]
      full   = M0

    如果想改成别的 prefix_rows，可以传入一个整数，表示取前 prefix_rows 行作 prefix
    """
    M0 = specialize_matrix_numeric(M, pt)

    if prefix_rows is None:
        prefix_rows = M0.nrows() - 1

    P0 = M0[:prefix_rows, :]
    r_pref = P0.rank()
    r_full = M0.rank()
    gap = r_full - r_pref

    out = {
        "rank_prefix": r_pref,
        "rank_full": r_full,
        "gap": gap,
        "ok": (gap == 0),
    }
    if return_matrix:
        out["matrix"] = M0
    return out


def random_search_rank_drop(
    M,
    max_trials=1000,
    fixed_assignments=None,
    free_vars=None,
    value_pool=None,
    prefix_rows=None,
    stop_after_first_good=True,
    seed=0,
    progress_every=100,
    verbose=True,
    keep_good_matrices=False,
):
    """
    直接随机搜点:
      - M: 例如 Ms[0]
      - max_trials: 随机次数
      - fixed_assignments: 固定某些 t
      - free_vars: 保留接口，不传即可
      - value_pool: 例如 range(50)；不传则在整个底域随机
      - prefix_rows: 默认 nrows-1，即检查“最后一行是否被前面 span 到”
      - stop_after_first_good: 找到一个 gap=0 就停
      - keep_good_matrices: 是否把命中的数值矩阵也存下来

    返回统计 summary
    """
    if seed is not None:
        random.seed(int(seed))

    stats = {
        "tested_points": 0,
        "gap_hist": {},          # 统计 gap 分布
        "best_gap": None,        # 最小 gap
        "first_good_point": None,
        "num_good_points": 0,
    }

    good_points = []

    t0 = time.time()

    for trial in range(1, max_trials + 1):
        pt = random_point_for_matrix(
            M,
            fixed_assignments=fixed_assignments,
            free_vars=free_vars,
            value_pool=value_pool
        )

        info = rank_gap_at_point(M, pt, prefix_rows=prefix_rows, return_matrix=keep_good_matrices)
        gap = info["gap"]

        stats["tested_points"] += 1
        stats["gap_hist"][gap] = stats["gap_hist"].get(gap, 0) + 1

        if stats["best_gap"] is None or gap < stats["best_gap"]:
            stats["best_gap"] = gap

        if verbose and progress_every is not None and trial % progress_every == 0:
            elapsed = time.time() - t0
            print(f"[trial {trial}] elapsed={elapsed:.2f}s, best_gap={stats['best_gap']}, gap_hist={stats['gap_hist']}")

        if info["ok"]:
            stats["num_good_points"] += 1
            if stats["first_good_point"] is None:
                stats["first_good_point"] = dict(pt)

            rec = {
                "point": dict(pt),
                "rank_prefix": info["rank_prefix"],
                "rank_full": info["rank_full"],
                "gap": info["gap"],
            }
            if keep_good_matrices:
                rec["matrix"] = info["matrix"]

            good_points.append(rec)

            if verbose:
                elapsed = time.time() - t0
                print("\nFOUND GOOD POINT")
                print("trial        =", trial)
                print("elapsed      =", f"{elapsed:.2f}s")
                print("rank_prefix  =", info["rank_prefix"])
                print("rank_full    =", info["rank_full"])
                print("gap          =", info["gap"])
                print("point        =", pt)

            if stop_after_first_good:
                break

    elapsed = time.time() - t0

    summary = {
        "stats": stats,
        "good_points": good_points,
        "elapsed": elapsed,
    }

    if verbose:
        print("\n========== random search summary ==========")
        print("tested_points     =", stats["tested_points"])
        print("gap_hist          =", stats["gap_hist"])
        print("best_gap          =", stats["best_gap"])
        print("num_good_points   =", stats["num_good_points"])
        print("first_good_point  =", stats["first_good_point"])
        print("elapsed           =", f"{elapsed:.2f}s")

    return summary

def random_univariate_slice_from_macaulay(
    M,
    keep_var=None,
    trials=50,
    value_pool=None,
    seed=None,
    verbose=True,
):
    """
    输入:
        M         : Macaulay矩阵 (entries 在某个多项式环里)
        keep_var  : 保留为唯一未知量的变量。可以传 ring 的 generator，
                    也可以传变量名字符串；若为 None，默认保留最后一个变量
        trials    : 随机切片次数
        value_pool: 若底层不是有限域，可指定一个随机值池，如 list(range(10))
        seed      : 随机种子
        verbose   : 是否打印过程

    返回:
        一个列表 results，每个元素对应一次 trial 的结果字典。
    """

    if seed is not None:
        set_random_seed(seed)

    # 原多项式环
    R0 = M.base_ring()
    base = R0.base_ring()
    orig_gens = list(R0.gens())
    nrows = M.nrows()
    ncols = M.ncols()

    # ---- 构造扩张多项式环：原变量 + c变量 ----
    c_names = [f'c{i}' for i in range(nrows)]
    all_names = [str(x) for x in orig_gens] + c_names
    R = PolynomialRing(base, all_names, order='degrevlex')

    # 迁移矩阵到新环
    M_R = M.change_ring(R)

    # 原变量与新环变量的对应
    new_orig = list(R.gens()[:len(orig_gens)])
    c_vars = list(R.gens()[len(orig_gens):])

    # 选保留变量
    if keep_var is None:
        keep = R.gens()[-1]
    elif isinstance(keep_var, str):
        keep = R(keep_var)
    else:
        keep_name = str(keep_var)
        keep = R(keep_name)

    all_vars = list(R.gens())
    if keep not in all_vars:
        raise ValueError(f"keep_var={keep} 不在变量列表中")

    # ---- 构造 c*M 的列方程 ----
    # eq_j = sum_i c_i * M[i,j]
    eqs = []
    for j in range(ncols):
        eq = sum(c_vars[i] * M_R[i, j] for i in range(nrows))
        if eq != 0:
            eqs.append(eq)

    if verbose:
        print("====================================")
        print("构造出的列方程数量:", len(eqs))
        print("总变量数:", len(all_vars))
        print("变量列表:", all_vars)
        print("保留变量:", keep)
        print("====================================")

    # ---- 随机赋值函数 ----
    def random_value():
        # 有限域：直接随机域元素
        if hasattr(base, "is_finite") and base.is_finite():
            elems = list(base)
            return elems[ZZ.random_element(0, len(elems))]
        # 其他情况：从 value_pool 里抽
        if value_pool is None:
            return ZZ.random_element(-5, 6)
        return value_pool[ZZ.random_element(0, len(value_pool))]

    # ---- 单变量系统求解 ----
    results = []

    for tr in range(trials):
        subs = {}
        for v in all_vars:
            if v != keep:
                subs[v] = random_value()

        sliced = [f.subs(subs) for f in eqs]
        sliced = [f for f in sliced if f != 0]

        # 检查是否真的都变成 keep 的单变量多项式
        bad = []
        for f in sliced:
            vars_f = f.variables()
            if any(v != keep for v in vars_f):
                bad.append(f)

        if bad:
            result = {
                'trial': tr,
                'status': 'not_univariate_after_subs',
                'subs': subs,
                'bad_equations': bad[:5],
            }
            results.append(result)
            if verbose:
                print(f"[trial {tr}] 代入后仍非单变量，跳过")
            continue

        # 全部恒零
        if len(sliced) == 0:
            result = {
                'trial': tr,
                'status': 'all_zero',
                'subs': subs,
                'has_root': True,   # 此时对 keep 没有限制
                'roots': 'free variable',
            }
            results.append(result)
            if verbose:
                print(f"[trial {tr}] 所有方程都变成 0，{keep} 自由")
            continue

        # 放到单变量环里做 gcd
        S = PolynomialRing(base, str(keep))
        u = S.gen()

        polys = []
        for f in sliced:
            fS = S(f)
            if fS != 0:
                polys.append(fS)

        if len(polys) == 0:
            result = {
                'trial': tr,
                'status': 'all_zero_after_cast',
                'subs': subs,
                'has_root': True,
                'roots': 'free variable',
            }
            results.append(result)
            if verbose:
                print(f"[trial {tr}] 转到单变量环后全 0，{keep} 自由")
            continue

        g = polys[0]
        for f in polys[1:]:
            g = gcd(g, f)

        if verbose:
            print(f"\n[trial {tr}]")
            print("固定变量赋值:")
            print(subs)
            print("剩余单变量多项式个数:", len(polys))
            print("gcd =", g)

        # gcd=1 表示公共根为空
        if g.degree() <= 0:
            result = {
                'trial': tr,
                'status': 'no_common_root',
                'subs': subs,
                'has_root': False,
                'gcd': g,
                'polys': polys[:10],
            }
            results.append(result)
            if verbose:
                print("=> 无公共根")
            continue

        # 找根
        roots = []
        try:
            # 有限域/可分解情形
            fac = g.factor()
            for h, e in fac:
                if h.degree() == 1:
                    # h = a*u + b
                    a = h[u.degree()] if False else None  # 占位，不使用
            roots = g.roots(multiplicities=False)
        except Exception:
            fac = None
            try:
                roots = g.roots()
            except Exception:
                roots = "unable to compute directly"

        result = {
            'trial': tr,
            'status': 'common_root_exists',
            'subs': subs,
            'has_root': True,
            'gcd': g,
            'factorization': fac,
            'roots': roots,
            'polys': polys[:10],
        }
        results.append(result)

        if verbose:
            print("=> 可能有公共根")
            print("roots =", roots)

    return results

def solve_ct_system_gb(
    M,
    nonzero_vars=('t1', 't3', 't5'),
    normalize_c_index=0,
    fixed_vars=None,
    max_slice_trials=50,
    seed=1234,
    verbose=True,
):
    """
    直接构造 c,t 联合系统并求 Gröbner basis。

    输入:
        M                : Macaulay矩阵，例如 Ms[0]
        nonzero_vars     : 必须非零的参数名列表/元组，如 ('t1','t3','t5')
        normalize_c_index: 固定哪个 c_i = 1 来排除平凡解
        fixed_vars       : 预先固定的变量，字典，如 {'t1': 1} 或 {T[1]: 1}
        max_slice_trials : 若理想非0维，随机切片找点的最大次数
        seed             : 随机种子
        verbose          : 是否打印过程

    返回:
        dict，包含：
            ring
            equations
            ideal
            groebner_basis
            dimension
            solution / solutions / sliced_solution
    """

    if fixed_vars is None:
        fixed_vars = {}

    if seed is not None:
        set_random_seed(seed)

    R0 = M.base_ring()
    K = R0.base_ring()

    # ---- 只收集 M 中实际出现的原变量 ----
    used_var_set = set()
    for i in range(M.nrows()):
        for j in range(M.ncols()):
            f = M[i, j]
            if f != 0:
                used_var_set.update(f.variables())

    used_vars = sorted(used_var_set, key=str)
    used_names = [str(v) for v in used_vars]

    if verbose:
        print("========================================")
        print("M size =", M.nrows(), "x", M.ncols())
        print("used variables =", used_names)
        print("number of used variables =", len(used_names))

    # ---- 新变量：c0,...,c_{m-1} 和辅助变量 z ----
    m = M.nrows()
    c_names = [f'c{i}' for i in range(m)]
    z_name = 'z_nonzero_aux'

    all_names = used_names + c_names + [z_name]

    # 用 lex 更适合后面消元/解
    R = PolynomialRing(K, all_names, order='lex')

    # 旧变量映到新环
    sub_to_R = {v: R(str(v)) for v in used_vars}

    # 迁移矩阵到新环
    M_new = matrix(
        R,
        M.nrows(),
        M.ncols(),
        [
            R(M[i, j].subs(sub_to_R)) if M[i, j] != 0 else R.zero()
            for i in range(M.nrows())
            for j in range(M.ncols())
        ]
    )

    orig_vars_R = [R(name) for name in used_names]
    c_vars = [R(name) for name in c_names]
    z = R(z_name)

    # ---- 处理 fixed_vars：统一转成新环里的变量 ----
    fixed_subs = {}
    for k, v in fixed_vars.items():
        k_str = str(k)
        if k_str not in all_names:
            raise ValueError(f"fixed_vars 中的 {k} 不在系统变量里。可选变量: {all_names}")
        fixed_subs[R(k_str)] = R(v)

    # 已固定变量名
    fixed_names = {str(k) for k in fixed_vars.keys()}

    # ---- 构造列方程：c * M = 0 ----
    eqs = []
    for j in range(M_new.ncols()):
        eq = sum(c_vars[i] * M_new[i, j] for i in range(M_new.nrows()))
        if eq != 0:
            eqs.append(eq)

    # ---- 先代入 fixed_vars ----
    if fixed_subs:
        eqs = [eq.subs(fixed_subs) for eq in eqs]

    # ---- 加归一化条件：c_k = 1 ----
    if normalize_c_index < 0 or normalize_c_index >= len(c_vars):
        raise ValueError("normalize_c_index 超出范围")
    eqs.append(c_vars[normalize_c_index] - 1)

    # ---- fixed 后，nonzero 中已固定的变量不用再乘进去 ----
    effective_nonzero_vars = tuple(name for name in nonzero_vars if str(name) not in fixed_names)

    # ---- 加非零条件：z * prod(nonzero_vars) - 1 = 0 ----
    nz_prod = R.one()
    for name in effective_nonzero_vars:
        if str(name) not in all_names:
            raise ValueError(f"nonzero_vars 中的 {name} 不在系统变量里。可选变量: {all_names}")
        nz_prod *= R(str(name))
    eqs.append(z * nz_prod - 1)

    # fixed_subs 也要作用到后加的方程（虽然通常不会影响 c_k=1 和 z*prod-1）
    if fixed_subs:
        eqs = [eq.subs(fixed_subs) for eq in eqs]

    # 去掉恒等于 0 的方程
    eqs = [eq for eq in eqs if eq != 0]

    if verbose:
        print("number of equations =", len(eqs))
        print("nonzero constraints on =", list(effective_nonzero_vars))
        print("normalization =", c_vars[normalize_c_index], "= 1")
        print("fixed vars =", {str(k): v for k, v in fixed_vars.items()})
        print("total variables in ring =", len(all_names))
        print("ring =", R)
        print("========================================")

    I = Ideal(eqs)

    # ---- 先尝试看维数 ----
    try:
        dimI = I.dimension()
    except Exception as e:
        dimI = None
        if verbose:
            print("dimension() failed:", e)

    if verbose:
        print("Ideal dimension =", dimI)
        print("Computing Gröbner basis ...")

    # ---- 直接算 GB ----
    try:
        G = I.groebner_basis()
        if verbose:
            print("GB computed, size =", len(G))
    except Exception as e:
        if verbose:
            print("groebner_basis() failed:", e)
        return {
            'ring': R,
            'equations': eqs,
            'ideal': I,
            'groebner_basis': None,
            'dimension': dimI,
            'error': e,
        }

    result = {
        'ring': R,
        'equations': eqs,
        'ideal': I,
        'groebner_basis': G,
        'dimension': dimI,
    }

    # =========================================================
    # 0维：直接尝试找根
    # =========================================================
    if dimI == 0:
        if verbose:
            print("Ideal appears 0-dimensional. Trying to solve ...")

        try:
            sols = I.variety()
            if verbose:
                print("number of solutions =", len(sols))

            clean_sols = []
            for sol in sols:
                clean = {str(k): v for k, v in sol.items() if str(k) != z_name}

                # 把 fixed_vars 也补回输出，便于看完整解
                for k, v in fixed_vars.items():
                    clean[str(k)] = v

                clean_sols.append(clean)

            result['solutions'] = clean_sols

            if len(clean_sols) > 0:
                result['solution'] = clean_sols[0]
                if verbose:
                    print("Found a solution:")
                    print(clean_sols[0])
            return result

        except Exception as e:
            if verbose:
                print("variety() failed:", e)
                print("Will return GB only.")
            result['solve_error'] = e
            return result

    # =========================================================
    # 非0维：随机切片找一个点
    # =========================================================
    if verbose:
        print("Ideal is not 0-dimensional (or dimension unknown).")
        print("Trying random affine slices to get one point ...")

    # 已固定变量不要再参与切片
    orig_vars_free = [v for v in orig_vars_R if str(v) not in fixed_names]
    vars_for_slice = [v for v in orig_vars_free + c_vars if v != c_vars[normalize_c_index]]
    # z 不切，留给非零约束自己处理

    def randval():
        if hasattr(K, "is_finite") and K.is_finite():
            L = list(K)
            return L[ZZ.random_element(len(L))]
        return ZZ.random_element(-5, 6)

    for tr in range(max_slice_trials):
        subs = {}
        free_keep = []

        # 默认留 1 个未固定原变量
        if len(orig_vars_free) > 0:
            keep_var = orig_vars_free[ZZ.random_element(len(orig_vars_free))]
            free_keep = [keep_var]
        else:
            free_keep = []

        for v in vars_for_slice:
            if v not in free_keep:
                subs[v] = randval()

        sliced_eqs = [f.subs(subs) for f in eqs]
        sliced_eqs = [f for f in sliced_eqs if f != 0]
        J = Ideal(sliced_eqs)

        try:
            dJ = J.dimension()
        except Exception:
            dJ = None

        if verbose:
            print(f"[slice {tr}] keep =", [str(v) for v in free_keep], "dimension =", dJ)

        try:
            if dJ == 0:
                sols = J.variety()
                if len(sols) > 0:
                    sol0 = sols[0]
                    full_sol = {}

                    for name in all_names:
                        vv = R(name)
                        if vv in sol0:
                            full_sol[name] = sol0[vv]
                        elif vv in subs:
                            full_sol[name] = subs[vv]

                    # 补回 fixed_vars
                    for k, v in fixed_vars.items():
                        full_sol[str(k)] = v

                    if z_name in full_sol:
                        del full_sol[z_name]

                    result['sliced_solution'] = full_sol
                    if verbose:
                        print("Found sliced solution:")
                        print(full_sol)
                    return result
        except Exception as e:
            if verbose:
                print(f"[slice {tr}] solve failed:", e)

    if verbose:
        print("No sliced solution found.")
    return result

In [4]:
# from sage.all import *
# import random

# class PoseidonSage:
#     def __init__(self, t=5, rate=4, capacity=1, s=1, p=101, alpha=3, n_rounds=30, seed=None):
#         """
#         Poseidon toy model in Sage
#         """
#         assert rate + capacity == t, "rate + capacity must equal t"

#         self.t = t
#         self.rate = rate
#         self.capacity = capacity
#         self.s = s
#         self.p = p
#         self.alpha = alpha
#         self.n_rounds = n_rounds
#         self.seed = seed
#         self.F = GF(p)

#         if seed is not None:
#             random.seed(seed)

#         # round constants and MDS matrix
#         self.round_constants = self._generate_round_constants()
#         self.mds_matrix = self._generate_cauchy_mds()

#     # ------------------ Round constants ------------------
#     def _generate_round_constants(self):
#         RC = []
#         for _ in range(self.n_rounds):
#             RC.append(vector(self.F, [self.F(random.randint(0, self.p-1)) for _ in range(self.t)]))
#         return RC

#     # ------------------ Cauchy MDS matrix ------------------
#     def _generate_cauchy_mds(self):
#         elems = random.sample(range(self.p), 2*self.t)
#         x = elems[:self.t]
#         y = elems[self.t:]
#         M = matrix(self.F, self.t, self.t)
#         for i in range(self.t):
#             for j in range(self.t):
#                 diff = x[i] - y[j]
#                 assert diff != 0, "Zero denominator in Cauchy construction"
#                 M[i,j] = self.F(1)/self.F(diff)
#         return M

#     # ------------------ S-box ------------------
#     def sbox(self, x):
#         return x ** self.alpha

#     def external_sbox_layer(self, state):
#         return vector(self.F, [self.sbox(x) for x in state])

#     def internal_sbox_layer(self, state):
#         state = vector(state)
#         for i in range(self.s):
#             state[i] = self.sbox(state[i])
#         return state

#     def inv_sbox(self, x):
#         alpha_inv = inverse_mod(self.alpha, self.p - 1)
#         return x ** alpha_inv

#     def inv_internal_sbox_layer(self, state):
#         state = vector(state)
#         for i in range(self.s):
#             state[i] = self.inv_sbox(state[i])
#         return state
    
#     def inv_mds_layer(self, state):
#         return self.mds_matrix.inverse() * state

#     def sub_round_constants(self, state, round_idx):
#         return state - self.round_constants[round_idx]
#     # def sub_round_constants(self, state, round_idx):
#     #     R = state[0].parent()
#     #     rc = vector(R, [R(c) for c in self.round_constants[round_idx]])
#     #     return state - rc

#     def inv_internal_round(self, state, round_idx):
#         state = self.inv_mds_layer(state)
#         state = self.inv_internal_sbox_layer(state)
#         state = self.sub_round_constants(state, round_idx)
#         return state
#     # ------------------ Linear layer ------------------
#     def mds_layer(self, state):
#         return self.mds_matrix * state

#     # ------------------ Round functions ------------------
#     def add_round_constants(self, state, round_idx):
#         return state + self.round_constants[round_idx]

#     def external_round(self, state, round_idx):
#         state = self.add_round_constants(state, round_idx)
#         state = self.external_sbox_layer(state)
#         state = self.mds_layer(state)
#         return state

#     def internal_round(self, state, round_idx):
#         state = self.add_round_constants(state, round_idx)
#         state = self.internal_sbox_layer(state)
#         state = self.mds_layer(state)
#         return state

#     # ------------------ Kernel / Subspace search ------------------
#     def kernel_mod_p(self, M):
#         """
#         Compute kernel of M over GF(p)
#         Return as a matrix whose columns are basis vectors
#         """
#         return M.right_kernel().matrix().transpose()

#     def compute_mds_invariant_subspaces_sage(self):
#         p = self.p
#         s = self.s
#         Fp = GF(p)

#         # MDS matrix mod p
#         M = Matrix(ZZ, self.mds_matrix).apply_map(lambda x: x % p).change_ring(Fp)

#         B = M[:s, s:]
#         D = M[s:, s:]

#         D_inv = D.inverse()

#         # 初始不变子空间 W：n × k 矩阵（列空间）
#         W = self.kernel_mod_p(B).change_ring(Fp)
#         W_list = [W]

#         while W.ncols() > 0:
#             # W2 = D^{-1} W
#             W2 = D_inv * W

#             # Z = [ W | -W2 ]
#             Z = W.augment(-W2)

#             # 求核
#             ns = self.kernel_mod_p(Z).change_ring(Fp)
#             if ns.ncols() == 0:
#                 break

#             # W 的列向量空间
#             n = W.nrows()
#             k_old = W.ncols()

#             # 把 ns 的前 k_old 行当作系数
#             new_vectors = []
#             for i in range(ns.ncols()):
#                 coeffs = vector(Fp, [ns[j, i] for j in range(W.ncols())])
#                 v = W * coeffs          # v 是 n×1 矩阵
#                 new_vectors.append(v)  # 转成 vector

#             if not new_vectors:
#                 break

#             # 重新组装成 n × k_new 的矩阵
#             W = matrix(Fp, new_vectors).transpose()
#             W_list.append(W)

#         return W_list

#     def show_invariant_subspace_dynamics(self, W_list):
#         p = self.p
#         s = self.s
#         M = Matrix(ZZ, self.mds_matrix).apply_map(lambda x: x % p)
#         n = M.nrows()

#         for idx, W in enumerate(W_list, start=1):
#             print(f"\n{'='*16} W_{idx} {'='*16}")
#             print(f"Dimension: {W.ncols()}")
#             print("Basis (columns):")
#             print(W)

#             # padding s行零在上面
#             Z = zero_matrix(ZZ, s, W.ncols())
#             V = Z.stack(W).apply_map(lambda x: x % p)

#             print("\nInitial vectors after padding zeros:")
#             print(V)

#             # 连续左乘 M，次数等于 idx
#             cur = V
#             for step in range(1, idx + 1):
#                 cur = (M * cur).apply_map(lambda x: x % p)
#                 print(f"\nAfter multiplying M^{step}:")
#                 print(cur)


#         # ------------------ CICO equations over GF(p) ------------------
    
#     def make_symbolic_X(self, k):
#         """
#         返回 (R, X)
#         R: PolynomialRing
#         X: k维多项式向量
#         """
#         Fp = self.F
#         R = PolynomialRing(Fp, [f"x{i}" for i in range(k)])
#         X = vector(R, R.gens())
#         return R, X
    
#     def make_symbolic_X_with_t(self, k, num_t):
#         Fp = self.F

#         x_vars = [f"x{i}" for i in range(k)]
#         t_vars = [f"t{i}" for i in range(num_t)]

#         R = PolynomialRing(Fp, x_vars + t_vars)

#         gens = R.gens()

#         X = vector(R, gens[:k])
#         T = vector(R, gens[k:k+num_t])

#         return R, X, T

#     def initial_state_from_W(self, W, X):
#         """
#         state = [0,...,0 || W * X]
#         返回多项式向量（长度 t）
#         """
#         R = X[0].parent()
#         s = self.s

#         lin_part = list(W.change_ring(R) * X)   # 这是 list，不是 vector
#         state_list = [R.zero()] * s + lin_part

#         return vector(R, state_list)
    
#     def rp_round_polynomial(self, state, round_index):
#         """
#         一轮 internal round（r_p）
#         state: Polynomial vector
#         """
#         R = state[0].parent()
#         Fp = self.F
#         s  = self.s

#         # 加轮常量
#         rc = vector(R, [R(c) for c in self.round_constants[round_index]])
#         state = state + rc

#         # internal S-box
#         state = vector(R, state)
#         for i in range(s):
#             state[i] = state[i] ** self.alpha

#         # MDS
#         M = self.mds_matrix.change_ring(R)
#         state = M * state

#         return state
#     def rf_round_polynomial(self, state, round_index):
#         """
#         一轮 external round（r_f）
#         """
#         R = state[0].parent()

#         # 加轮常量
#         rc = vector(R, [R(c) for c in self.round_constants[round_index]])
#         state = state + rc

#         # external S-box
#         state = vector(R, [x ** self.alpha for x in state])

#         # MDS
#         M = self.mds_matrix.change_ring(R)
#         state = M * state

#         return state
#     # def generate_cico_equations_modp(self, W, constants, rounds, round_type="internal"):
#     def generate_cico_equations_modp(self, W, constants, rounds, exter1, inter, exter2):
#         """
#         round_type: "internal" or "external"
#         """
#         k = W.ncols()
#         R, X = self.make_symbolic_X(k)

#         # 初始状态
#         state = self.initial_state_from_W(W, X)

#         assert exter1 + inter + exter2 == rounds
#         round_idx = 0

#         # 轮函数迭代#############注意这里的轮常数序列是错的
#         for _ in range(exter1):
#             state = self.rf_round_polynomial(state, round_idx)
#             round_idx += 1
#         for _ in range(inter):
#             state = self.rp_round_polynomial(state, round_idx)
#             round_idx += 1
#         for _ in range(exter2):
#             state = self.rf_round_polynomial(state, round_idx)
#             round_idx += 1

#         # for r in range(rounds):
#         #     if round_type == "internal":
#         #         state = self.rp_round_polynomial(state, r)
#         #     else:
#         #         state = self.rf_round_polynomial(state, r)

#         # 构造 CICO 方程
#         const_vec = vector(R, [R(c) for c in constants])
#         equations = list(state - const_vec)

#         return equations, R, X

#     def generate_cico_equations_withnosubspace(self, constants, rounds, exter1, inter, exter2):
#         """
#         round_type: "internal" or "external"
#         """
#         k = self.t
#         R, X = self.make_symbolic_X(k)
#         W = identity_matrix(self.F, self.t)
#         # lift 到多项式环
#         state = W.change_ring(R) * X

#         assert exter1 + inter + exter2 == rounds
#         round_idx = 0

#         # 轮函数迭代#############注意这里的轮常数序列是错的
#         for _ in range(exter1):
#             state = self.rf_round_polynomial(state, round_idx)
#             round_idx += 1
#         for _ in range(inter):
#             state = self.rp_round_polynomial(state, round_idx)
#             round_idx += 1
#         for _ in range(exter2):
#             state = self.rf_round_polynomial(state, round_idx)
#             round_idx += 1

#         # for r in range(rounds):
#         #     if round_type == "internal":
#         #         state = self.rp_round_polynomial(state, r)
#         #     else:
#         #         state = self.rf_round_polynomial(state, r)

#         # 构造 CICO 方程
#         const_vec = vector(R, [R(c) for c in constants])
#         equations = list(state - const_vec)

#         return equations, R, X

#     def simple_form(self, target, rounds):
#         state = target
#         for round_idx in range(rounds):
#             state = self.inv_internal_round(state, round_idx)
#         return state
    
#     def retain_dim_shear_nosubspace(self, constants):######在S盒之前=t引入参数
#         k = 3
#         num_t = 16#######加一个高次参数
#         # R, X, t= self.make_symbolic_X_with_t(k)
#         R, X, T = self.make_symbolic_X_with_t(k, num_t)

#         # state = vector(R, [T[19]*X[0]+T[0], 
#         #                    T[20]*X[1]+T[1],
#         #                    T[21]*X[2]+T[2],
#         #                    T[22]*X[3]+T[3],
#         #                    R.zero()])
#         state = vector(R, [T[0], 
#                             T[1]*X[0]+T[2],
#                             T[3]*X[1]+T[4],
#                             T[5]*X[2]+T[6],
#                             R.zero()])
#         round_idx = 0

#         # 轮函数迭代#############注意这里的轮常数序列是错的
#         eqs = []
#         reduced_ideal = []

#         state = self.rp_round_polynomial(state, round_idx)
#         round_idx += 1
#         f = R(state[0])
#         eq = f - T[7]
#         eqs.append(eq)
#         reduced_ideal.append(eq)
#         state = vector(R, [T[7]] + list(state[1:]))

#         state = self.rp_round_polynomial(state, round_idx)
#         round_idx += 1
#         state = self.rp_round_polynomial(state, round_idx)
#         round_idx += 1
#         f = R(state[0])
#         eq = f - (T[8]*X[0]+T[9]*X[1]+T[10]*X[2]+T[11])
#         eqs.append(eq)
#         reduced_ideal.append(eq)
#         state = vector(R, [T[8]*X[0]+T[9]*X[1]+T[10]*X[2]+T[11]] + list(state[1:]))

#         # state = self.rp_round_polynomial(state, round_idx)
#         # round_idx += 1
#         # f = R(state[0])
#         # eq = f - (T[15]*X[0]+T[16]*X[1]+T[17]*X[2]+T[14])
#         # eqs.append(eq)
#         # reduced_ideal.append(eq)
#         # state = vector(R, [T[15]*X[0]+T[16]*X[1]+T[17]*X[2]+T[14]] + list(state[1:]))

#         state = self.rp_round_polynomial(state, round_idx)
#         round_idx += 1
#         # state = self.rp_round_polynomial(state, round_idx)
#         # round_idx += 1

#         # 构造 CICO 方程
#         # const_vec = vector(R, [R(c) for c in constants])
#         # eqs += list(state - const_vec)
#         target_state = vector(R, [
#             T[12]*X[0] + T[13]*X[1] + T[14]*X[2] + T[15],
#             R(constants[1]),
#             R(constants[2]),
#             R(constants[3]),
#             R(constants[4]),
#         ])
#         eqs += list(state - target_state)

#         return eqs, reduced_ideal, R, X, T
    
# def square_submatrix(M):
#     n = M.ncols()
#     return M[:n, :]

In [5]:
def x_degree(f, x_vars):
    """
    只统计多项式 f 在 x_vars 上的最大次数，不计 t 变量次数
    """
    if f == 0:
        return -1

    dmax = 0
    for exp in f.exponents():
        d = sum(exp[i] for i in range(len(x_vars)))
        if d > dmax:
            dmax = d
    return dmax

def coeff_only_t(f, mon, x_vars, S):
    """
    从 f 中提取 x-单项式 mon 的系数，系数中允许含 t 变量。

    例如：
        f = (t0+1)*x0^2 + (t0^2+t1)*x0*x1 + 3*t1*x1^2
    若 mon = x0^2
    返回:
        t0 + 1
    """

    R = f.parent()
    gens = R.gens()

    x_vars = list(x_vars)
    x_idx = [gens.index(x) for x in x_vars]
    t_idx = [i for i in range(len(gens)) if i not in x_idx]

    # t 变量子环
    T = S.gens()
    # t_names = [str(gens[i]) for i in t_idx]
    # S = PolynomialRing(R.base_ring(), t_names)
    # T = S.gens()

    # mon 在 x 变量上的目标指数
    target_exp = tuple(mon.degree(x) for x in x_vars)

    res = S.zero()

    for exp, coeff in f.dict().items():
        x_exp = tuple(exp[i] for i in x_idx)

        # 必须和列单项式完全匹配
        if x_exp == target_exp:
            term = S(coeff)
            for j, idx in enumerate(t_idx):
                if exp[idx] != 0:
                    term *= T[j] ** exp[idx]
            res += term

    return res
import time 

def build_macaulay_matrix(eqs, reduced_eqs, n, mult_deg, col_deg,
                          f3_index=-1,
                          lift_f3=True,
                          homogeneous=True):
    R = eqs[0].parent()
    X = list(R.gens()[:n])

    gens = R.gens()
    t_vars = list(gens[n:])
    S = PolynomialRing(R.base_ring(), [str(v) for v in t_vars])

    f3 = eqs[f3_index]
    print("enter build_macaulay_matrix")
    # 列
    if homogeneous:
        col_mons = monomials_of_degree(R, X, col_deg)
        mult_mons = monomials_of_degree(R, X, mult_deg)
    else:
        col_mons = monomials_leq_degree(R, X, col_deg)
        mult_mons = monomials_leq_degree(R, X, mult_deg)
        
    print("got monomials", len(mult_mons), len(col_mons))

    # <f1,f2> 产生的行
    base_rows = []
    for m in mult_mons:
        for f in reduced_eqs:
            print("multiplying one row...")
            base_rows.append(m * f)
    print("base_rows done", len(base_rows))
    matrices = []

    # f3 不升次
    if not lift_f3:
        rows = list(base_rows)
        rows.append(f3)

        M = matrix(S, len(rows), len(col_mons))
        # for i, f in enumerate(rows):
            # for j, mon in enumerate(col_mons):
            #     M[i, j] = coeff_only_t(f, mon, X, S)
        for i, f in enumerate(rows):
            cmap = {}
            for mon_exp, coeff in f.dict().items():
                x_exp = tuple(mon_exp[:len(X)])
                t_exp = tuple(mon_exp[len(X):])

                term = S(coeff)
                for k, e in enumerate(t_exp):
                    if e != 0:
                        term *= S.gen(k)**e

                cmap[x_exp] = cmap.get(x_exp, S.zero()) + term

            for j, mon in enumerate(col_mons):
                exp = tuple(mon.degree(v) for v in X)
                M[i, j] = cmap.get(exp, S.zero())
        return [M], col_mons

    # f3 升次
    deg_f3_x = max(sum(mon.degree(x) for x in X) for mon in f3.monomials()) if f3 != 0 else -1
    lift_deg = col_deg - deg_f3_x

    if lift_deg < 0:
        raise ValueError(f"col_deg={col_deg} 小于 f3 的 x-次数 {deg_f3_x}")

    if lift_deg == 0:
        lift_mons = [R.one()]
    else:
        lift_mons = monomials_of_degree(R, X, lift_deg)

    for m in lift_mons:
        rows = list(base_rows)
        rows.append(m * f3)

        M = matrix(S, len(rows), len(col_mons))
        # t0 = time.time()
        # for i, f in enumerate(rows):
        #     # print("building row", i, "num_terms =", len(f.dict()))
        #     # t1 = time.time()
        #     for j, mon in enumerate(col_mons):
        #         M[i, j] = coeff_only_t(f, mon, X)
        #     # print("row", i, "done, time =", time.time() - t1)  
        # # print("total build time =", time.time() - t0)
        for i, f in enumerate(rows):
            cmap = {}
            for mon_exp, coeff in f.dict().items():
                x_exp = tuple(mon_exp[:len(X)])
                t_exp = tuple(mon_exp[len(X):])

                term = S(coeff)
                for k, e in enumerate(t_exp):
                    if e != 0:
                        term *= S.gen(k)**e

                cmap[x_exp] = cmap.get(x_exp, S.zero()) + term

            for j, mon in enumerate(col_mons):
                exp = tuple(mon.degree(v) for v in X)
                M[i, j] = cmap.get(exp, S.zero())
        matrices.append(M)

    return matrices, col_mons

def build_macaulay_matrix_fast(
    eqs,
    reduced_ideal,
    n,
    mult_deg,
    col_deg,
    f3_index=3,
    lift_f3=False,
    homogeneous=False
):

    R = eqs[0].parent()
    gens = list(R.gens())

    X = gens[:n]
    T = gens[n:]

    # ---------- columns ----------
    if homogeneous:
        col_mons = monomials_of_degree(R, X, col_deg)
        mult_mons = monomials_of_degree(R, X, mult_deg)
    else:
        col_mons = monomials_leq_degree(R, X, col_deg)
        mult_mons = monomials_leq_degree(R, X, mult_deg)

    col_exp = [tuple(m.degree(v) for v in X) for m in col_mons]
    col_index = {e:i for i,e in enumerate(col_exp)}

    mult_exp = [tuple(m.degree(v) for v in X) for m in mult_mons]

    # ---------- coefficient extraction ----------
    def extract_x_coeffs(f):

        d = {}

        for mon_exp, coeff in f.dict().items():

            x_exp = tuple(mon_exp[:n])
            t_exp = mon_exp[n:]

            # rebuild t polynomial
            term = coeff
            for i,e in enumerate(t_exp):
                if e:
                    term *= T[i]**e

            d[x_exp] = d.get(x_exp, R.zero()) + term

        return d

    reduced_maps = [extract_x_coeffs(f) for f in reduced_ideal]
    last_map = extract_x_coeffs(eqs[f3_index])

    rows = []

    # ---------- lifted rows ----------
    for beta in mult_exp:
        for cmap in reduced_maps:

            row = [R.zero()] * len(col_mons)

            for alpha, tpoly in cmap.items():

                new_exp = tuple(a+b for a,b in zip(alpha,beta))

                j = col_index.get(new_exp)
                if j is not None:
                    row[j] += tpoly

            rows.append(row)

    # ---------- last equation ----------
    row = [R.zero()] * len(col_mons)

    for alpha, tpoly in last_map.items():

        j = col_index.get(alpha)
        if j is not None:
            row[j] += tpoly

    rows.append(row)

    M = matrix(R, rows)

    return [M], col_mons

def build_macaulay_matrix_mixed_fast(
    eqs,
    reduced_ideal,
    n,
    mult_deg,
    col_deg,
    f3_index=2,
    lift_f3=False,
    homogeneous=False
):
    """
    支持 reduced_ideal 中不同生成元使用不同乘子次数的 Macaulay 矩阵构造。

    参数：
        eqs            : 原方程列表
        reduced_ideal  : 用来生成前缀行的小理想生成元列表，如 [f1, f2]
        n              : x 变量个数（前 n 个变量视为 x，其余视为 t）
        mult_deg       : 
            - 若为 int，则所有 reduced_ideal 生成元统一用这个次数
            - 若为 list/tuple，则第 i 个生成元用 mult_deg[i]
              例如 [2,0] 表示 f1 乘 <=2 次单项式，f2 只乘 1
        col_deg        : 列单项式最大次数（或恰次数，取决于 homogeneous）
        f3_index       : eqs 中“目标多项式”下标
        lift_f3        : 是否也对 f3 做升次
        homogeneous    :
            - True  : 用恰次数 monomials_of_degree
            - False : 用小于等于次数 monomials_leq_degree

    返回：
        [M], col_mons
    """

    R = eqs[0].parent()
    gens = list(R.gens())

    X = gens[:n]
    T = gens[n:]

    r = len(reduced_ideal)

    # ---- 统一 mult_deg 成列表 ----
    if isinstance(mult_deg, (list, tuple)):
        if len(mult_deg) != r:
            raise ValueError(f"mult_deg 长度应等于 len(reduced_ideal)={r}，当前是 {len(mult_deg)}")
        mult_deg_list = list(mult_deg)
    else:
        mult_deg_list = [mult_deg] * r

    # ---- columns ----
    if homogeneous:
        col_mons = monomials_of_degree(R, X, col_deg)
    else:
        col_mons = monomials_leq_degree(R, X, col_deg)

    col_exp = [tuple(m.degree(v) for v in X) for m in col_mons]
    col_index = {e: i for i, e in enumerate(col_exp)}

    # ---- coefficient extraction ----
    def extract_x_coeffs(f):
        d = {}
        for mon_exp, coeff in f.dict().items():
            x_exp = tuple(mon_exp[:n])
            t_exp = mon_exp[n:]

            term = coeff
            for i, e in enumerate(t_exp):
                if e:
                    term *= T[i] ** e

            d[x_exp] = d.get(x_exp, R.zero()) + term
        return d

    reduced_maps = [extract_x_coeffs(f) for f in reduced_ideal]
    f3_map = extract_x_coeffs(eqs[f3_index])

    rows = []

    # ---- 前缀行：每个生成元各自按自己的 mult_deg 升次 ----
    for cmap, d in zip(reduced_maps, mult_deg_list):
        if homogeneous:
            if d < 0:
                raise ValueError("homogeneous=True 时 mult_deg 不能为负")
            mult_mons = monomials_of_degree(R, X, d)
        else:
            if d < 0:
                raise ValueError("homogeneous=False 时 mult_deg 不能为负")
            mult_mons = monomials_leq_degree(R, X, d)

        mult_exp = [tuple(m.degree(v) for v in X) for m in mult_mons]

        for beta in mult_exp:
            row = [R.zero()] * len(col_mons)

            for alpha, tpoly in cmap.items():
                new_exp = tuple(a + b for a, b in zip(alpha, beta))
                j = col_index.get(new_exp)
                if j is not None:
                    row[j] += tpoly

            rows.append(row)

    # ---- f3 行 ----
    if not lift_f3:
        row = [R.zero()] * len(col_mons)
        for alpha, tpoly in f3_map.items():
            j = col_index.get(alpha)
            if j is not None:
                row[j] += tpoly
        rows.append(row)

    else:
        # 自动根据 f3 的 x-次数决定还要乘多少次
        deg_f3_x = max((sum(exp[:n]) for exp in eqs[f3_index].dict().keys()), default=-1)
        lift_deg = col_deg - deg_f3_x

        if lift_deg < 0:
            raise ValueError(f"col_deg={col_deg} 小于 f3 的 x-次数 {deg_f3_x}")

        if homogeneous:
            lift_mons = monomials_of_degree(R, X, lift_deg)
        else:
            lift_mons = monomials_leq_degree(R, X, lift_deg)

        lift_exp = [tuple(m.degree(v) for v in X) for m in lift_mons]

        for beta in lift_exp:
            row = [R.zero()] * len(col_mons)
            for alpha, tpoly in f3_map.items():
                new_exp = tuple(a + b for a, b in zip(alpha, beta))
                j = col_index.get(new_exp)
                if j is not None:
                    row[j] += tpoly
            rows.append(row)

    M = matrix(R, rows)
    return [M], col_mons


def make_square_and_det(M, keep_last=True, verbose=False):
    """
    M: 原矩阵
    keep_last: 是否保留最后一行（目标方程）
    返回: 方阵，行列式
    """
    rows, cols = M.nrows(), M.ncols()
    
    if rows < cols:
        print("矩阵行数不足，无法变成方阵")
        return None, None
    else: 
    # 先确定要保留的行
        if keep_last:
            # 保留最后一行
            keep_idx = [rows-1]  # 最后一行
            # 然后按顺序选够 cols-1 行
            for i in range(rows-1):
                if len(keep_idx) >= cols:
                    break
                keep_idx.insert(0, i)  # 从前面开始插入
        else:
            # 直接选前 cols 行
            keep_idx = list(range(cols))
        
        if verbose:
            print(f"保留的行索引: {keep_idx}")

        # 生成方阵
        M_square = M[keep_idx, :]
        
        # 计算行列式
        det_val = M_square.det()
        
        return M_square, det_val

def test_random_ranks(M, trials=10):
    S = M.base_ring()
    F = S.base_ring()
    ts = S.gens()

    for _ in range(trials):
        vals = [F.random_element() for _ in ts]
        M0 = matrix(F, M.substitute({ts[i]: vals[i] for i in range(len(ts))}))
        r1 = M0.rank()
        r2 = M0[:-1].rank()
        print("vals =", vals, " rank =", r1, " rank_without_last =", r2, " same?", r1 == r2)

def solve_t_for_last_row_span(M):
    """
    设前 r 行线性组合等于最后一行：
        c0*row0 + ... + c_{r-1}*row_{r-1} = last_row
    返回一个带消元顺序的多项式环，以及对应方程。
    """
    S = M.base_ring()          # 例如 GF(p)[t0,t1,t2]
    F = S.base_ring()
    t_names = [str(x) for x in S.gens()]

    r = M.nrows() - 1
    c_names = [f'c{i}' for i in range(r)]

    # 关键：消元顺序，先消 c，再留 t
    # c 在前，t 在后；lex 通常够用
    Rbig = PolynomialRing(F, c_names + t_names, order='lex')
    vars_big = Rbig.gens()

    nc = len(c_names)
    nt = len(t_names)

    c_big = vars_big[:nc]
    t_big = vars_big[nc:]

    # 把 S 中的 t 变量映到 Rbig 中的 t_big
    sub_t = {S.gens()[i]: t_big[i] for i in range(nt)}

    eqs = []
    for j in range(M.ncols()):
        lhs = Rbig.zero()
        for i in range(r):
            lhs += c_big[i] * Rbig(M[i, j].subs(sub_t))
        rhs = Rbig(M[r, j].subs(sub_t))
        eqs.append(lhs - rhs)

    return Rbig, eqs, c_big, t_big

def independent_row_basis(M):
    """
    对矩阵 M 的前 n-1 行，在分式域上取一组线性无关“行空间基”。
    这里返回的不是原始若干行，而是 echelon form 里的非零行，
    也就是允许线性组合后的基。

    返回:
        B : 前缀行空间的一组基（matrix over fraction field）
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    A = Mf[:-1, :]
    E = A.echelon_form()

    rows = []
    for i in range(E.nrows()):
        if E.row(i) != 0:
            rows.append(E.row(i))

    if len(rows) == 0:
        return matrix(K, 0, M.ncols())

    return matrix(K, rows)

def compress_rows_keep_last(M):
    """
    把前缀行压成一组线性无关基，再把最后一行接上。
    """
    S = M.base_ring()
    K = S.fraction_field()

    B = independent_row_basis(M)
    last = matrix(K, [matrix(K, M)[-1]])

    return B.stack(last)

def support_columns_of_matrix(M):
    """
    返回矩阵 M 中所有出现过非零元的列下标。
    """
    cols = []
    for j in range(M.ncols()):
        nonzero = False
        for i in range(M.nrows()):
            if M[i, j] != 0:
                nonzero = True
                break
        if nonzero:
            cols.append(j)
    return cols

def pivot_columns_of_prefix(M):
    """
    返回前面行（不含最后一行）在分式域上的 pivot 列
    """
    S = M.base_ring()
    K = S.fraction_field()
    A = matrix(K, M[:-1, :])
    piv = A.pivots()
    return list(piv)
def important_columns(M, extra=5):
    """
    取 pivot 列，再额外补一些最后一行非零列
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    piv = pivot_columns_of_prefix(M)
    last = Mf[-1]

    extra_cols = []
    for j in range(M.ncols()):
        if j not in piv and last[j] != 0:
            extra_cols.append(j)
        if len(extra_cols) >= extra:
            break

    cols = sorted(set(piv + extra_cols))
    return cols
def compress_matrix(M):
    """
    先压前缀行到线性无关基，再保留必要列。

    必要列 = 前缀基真正用到的列 ∪ 最后一行真正用到的列
    """
    M1 = compress_rows_keep_last(M)
    cols = safe_columns_for_last_row_test(M)
    return M1[:, cols], cols

def independent_row_indices_over_fraction_field(M):
    """
    对矩阵 M 的前 n-1 行，在分式域上找一组线性无关的行下标
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    A = Mf[:-1, :]
    idx = []
    current = matrix(K, 0, A.ncols())

    for i in range(A.nrows()):
        candidate = current.stack(matrix(K, [A.row(i)]))
        if candidate.rank() > current.rank():
            idx.append(i)
            current = candidate

    return idx

def t_conditions_from_last_row_span(M, pivot_cols_override=None, verbose=True):
    """
    目标：
        找 t 的条件，使最后一行属于“前面各行张成的行空间”。

    这里采用的是“显式解系数 c 再代回”的方法，不是 Groebner 消元版。

    做法：
    1. 对前缀行取一组线性无关基 B（允许线性组合）
    2. 选 B 的一组 pivot 列
    3. 在这些列上解出系数 c(t)
    4. 代回全部列，得到残差 residual_j(t)
    5. 取这些残差的分子，作为只关于 t 的候选条件

    返回:
        {
            "B": 前缀行空间的一组基
            "pivot_cols": 用来解 c 的列
            "c_sol": 解出的系数 c(t)
            "residuals": 所有列上的残差
            "numerators": 残差分子去重后的列表
        }

    注意：
    - 这是在某个 pivot minor 非零的开集上成立的条件。
    - 若该 minor 为 0，则这组 c(t) 的公式可能失效，需要换别的列块。
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    # 前缀行与最后一行
    A = Mf[:-1, :]
    b = vector(K, Mf[-1])

    # 新逻辑：前缀行空间的一组基（允许线性组合）
    B = independent_row_basis(M)

    if verbose:
        print("原前缀行数 =", A.nrows())
        print("前缀行空间维数 =", B.nrows())
        print("列数 =", B.ncols())

    r = B.nrows()
    n = B.ncols()

    if r == 0:
        # 前缀全是零行
        residuals = [ -b[j] for j in range(n) ]
        numerators = []
        for rj in residuals:
            if rj != 0:
                try:
                    num = rj.numerator()
                except AttributeError:
                    num = rj
                if num != 0:
                    numerators.append(num)

        numerators_unique = []
        seen = set()
        for g in numerators:
            s = str(g)
            if s not in seen:
                seen.add(s)
                numerators_unique.append(g)

        return {
            "B": B,
            "pivot_cols": [],
            "c_sol": vector(K, []),
            "residuals": residuals,
            "numerators": numerators_unique,
        }

    # 选 B 的 pivot 列：要从列里挑 r 列，使对应子块可逆
    if pivot_cols_override is None:
        piv = list(B.pivots())
        if len(piv) < r:
            raise ValueError("B 的 pivot 列数小于行秩，无法构造 c(t) 的解")
    else:
        piv = list(pivot_cols_override)
        if len(piv) != r:
            raise ValueError(f"手动指定的 pivot 列数必须等于前缀行空间维数 r={r}")
        if len(set(piv)) != r:
            raise ValueError("手动指定的 pivot 列里有重复")
        if min(piv) < 0 or max(piv) >= n:
            raise ValueError("手动指定的 pivot 列超出范围")

    if verbose:
        print("pivot 列 =", piv)

    # C 是 r x r 子块，要求可逆
    C = B.matrix_from_columns(piv)
    detC = C.det()

    if verbose:
        print("det(pivot block) =", detC)

    if detC == 0:
        raise ValueError("当前指定的 pivot 列对应子块不可逆，请换一组列")

    # 在 pivot 列上解：
    #   sum_i c_i * B[i, piv[k]] = b[piv[k]]
    # 也就是 (C^T) c = b_piv
    rhs = vector(K, [b[j] for j in piv])
    c_sol = C.transpose().solve_right(rhs)

    if verbose:
        print("解出的 c(t) =")
        for i, ci in enumerate(c_sol):
            print(f"c[{i}] = {ci}")

    # 代回所有列，求残差
    residuals = []
    for j in range(n):
        val = sum(c_sol[i] * B[i, j] for i in range(r)) - b[j]
        residuals.append(val)

    # 取分子
    numerators = []
    for rj in residuals:
        if rj != 0:
            try:
                num = rj.numerator()
            except AttributeError:
                num = rj
            if num != 0:
                numerators.append(num)

    # 去重
    numerators_unique = []
    seen = set()
    for g in numerators:
        s = str(g)
        if s not in seen:
            seen.add(s)
            numerators_unique.append(g)

    return {
        "B": B,
        "pivot_cols": piv,
        "pivot_det": detC,
        "c_sol": c_sol,
        "residuals": residuals,
        "numerators": numerators_unique,
    }

def print_t_conditions(info):
    nums = info["numerators"]
    print("只关于 t 的候选条件（来自残差分子）:")
    if len(nums) == 0:
        print("没有残差条件：在当前选的 pivot minor 非零时，最后一行已经被 span。")
    else:
        for i, g in enumerate(nums):
            print(f"[{i}] {g}")

def kernel_hits_last_row(M, verbose=True):
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    Kleft = Mf.transpose().right_kernel()
    basis = Kleft.basis()

    if verbose:
        print("kernel dimension =", Kleft.dimension())
        for idx, v in enumerate(basis):
            print(f"basis[{idx}] =", v)
            print("last entry =", v[-1])
            print("----")

    hit = any(v[-1] != 0 for v in basis)
    print("是否存在最后一项非零的核向量:", hit)
    return Kleft, hit

def safe_columns_for_last_row_test(M):
    """
    安全压列：

    1. 先对前缀行取一组线性无关基 B（允许线性组合）
    2. 保留 B 中所有真正出现过的列
    3. 再并上最后一行中所有非零列

    这样不会因为只保留 pivot 列而丢掉前缀行空间的信息。
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    B = independent_row_basis(M)
    last = Mf[-1]

    supp_B = support_columns_of_matrix(B)
    supp_last = [j for j in range(M.ncols()) if last[j] != 0]

    cols = sorted(set(supp_B + supp_last))
    return cols

def compress_rows_keep_last_only(M):
    """
    只对前缀行做压缩：
    - 前缀行变成一组线性无关基（允许线性组合）
    - 最后一行原样保留

    返回:
        Msmall : 压缩后的矩阵
        info   : 一些辅助信息
    """
    S = M.base_ring()
    K = S.fraction_field()
    Mf = matrix(K, M)

    B = independent_row_basis(M)
    last = matrix(K, [Mf[-1]])

    Msmall = B.stack(last)

    info = {
        "prefix_rank": B.nrows(),
        "ncols": M.ncols(),
        "basis_support_cols": support_columns_of_matrix(B),
        "last_row_support_cols": [j for j in range(M.ncols()) if Mf[-1, j] != 0],
    }

    return Msmall, info

def variables_in_polys(polys):
    """
    从多项式列表里提取真正出现过的变量。
    返回按字符串排序后的变量列表。
    """
    vars_set = set()
    for f in polys:
        try:
            vars_set.update(f.variables())
        except AttributeError:
            pass
    return sorted(vars_set, key=str)
def analyze_t_ideal_from_info(info, verbose=True):
    """
    对 info["numerators"] 生成的 t-理想做分析。

    返回:
        {
            "polys": ...,
            "vars": ...,
            "ideal": ...,
            "gb": ...,
            "dimension": ...,
            "is_zero_dimensional": ...,
        }
    """
    polys = [f for f in info["numerators"] if f != 0]

    if len(polys) == 0:
        print("没有任何 t-约束，多半表示在当前开集上自动成立。")
        return {
            "polys": [],
            "vars": [],
            "ideal": None,
            "gb": None,
            "dimension": None,
            "is_zero_dimensional": None,
        }

    R = polys[0].parent()
    tvars = variables_in_polys(polys)
    J = R.ideal(polys)

    if verbose:
        print("=== t-ideal basic info ===")
        print("number of generators =", len(polys))
        print("variables appearing =", tvars)
        print()

        print("generators:")
        for i, f in enumerate(polys):
            print(f"[{i}] {f}")
        print()

    # Groebner basis
    try:
        gb = J.groebner_basis()
    except Exception as e:
        print("Groebner basis 计算失败：", e)
        gb = None

    if verbose and gb is not None:
        print("Groebner basis:")
        for i, g in enumerate(gb):
            print(f"GB[{i}] {g}")
        print()

    # 维数判断
    dimJ = None
    is_zero_dim = None

    try:
        dimJ = J.dimension()
        is_zero_dim = (dimJ == 0)
    except Exception as e:
        if verbose:
            print("J.dimension() 失败：", e)
        # 退化判断：如果 Groebner basis 里每个变量都有某个纯变量首项，往往是 0 维
        if gb is not None:
            try:
                is_zero_dim = J.is_zero_dimensional()
            except Exception:
                is_zero_dim = None

    if verbose:
        print("dimension =", dimJ)
        print("is_zero_dimensional =", is_zero_dim)
        print()

    return {
        "polys": polys,
        "vars": tvars,
        "ideal": J,
        "gb": gb,
        "dimension": dimJ,
        "is_zero_dimensional": is_zero_dim,
    }

def analyze_t_ideal_small_ring(info, verbose=True):
    """
    适配当前 info 结构:
        info.keys() = ['B', 'pivot_cols', 'pivot_det', 'c_sol', 'residuals', 'numerators']

    做的事：
      1. 从 numerators 读取方程
      2. 从这些方程的 parent / 或 B 的 base ring 推出原环
      3. 自动识别名字以 t 开头的变量
      4. 建立只含 t 的小环
      5. 把 numerators 搬到小环并做 Groebner 分析
    """

    if "numerators" not in info:
        raise KeyError(f"info 里没有 'numerators'。现有 keys = {list(info.keys())}")

    numerators = info["numerators"]
    if len(numerators) == 0:
        raise ValueError("info['numerators'] 是空的，无法建立小理想")

    if verbose:
        print("=== info keys ===")
        print(list(info.keys()))
        print("number of numerators =", len(numerators))

    # 1) 推原环 R
    R = None

    # 最稳：先从 numerator parent 取
    try:
        R = numerators[0].parent()
    except Exception:
        R = None

    # 若上面失败，再尝试从 B 的 base ring 取
    if R is None and "B" in info:
        try:
            R = info["B"].base_ring()
        except Exception:
            R = None

    if R is None:
        raise ValueError("无法从 numerators 或 B 推断原多项式环")

    if verbose:
        print("\nDetected parent ring R:")
        print(R)

    # 2) 自动找 t 变量
    gens = list(R.gens())
    t_vars = [g for g in gens if str(g).startswith("t")]

    if len(t_vars) == 0:
        raise ValueError(f"在原环生成元里没有找到名字以 't' 开头的变量。gens = {gens}")

    if verbose:
        print("\nDetected t_vars:")
        print(t_vars)

    # 3) 建小环
    F = R.base_ring()
    Rsmall = PolynomialRing(F, [str(v) for v in t_vars], order='lex')
    us = Rsmall.gens()
    sub_map = {t_vars[i]: us[i] for i in range(len(t_vars))}

    if verbose:
        print("\nRsmall =")
        print(Rsmall)

    # 4) 把 numerators 搬到小环
    polys_small = []
    failed = []

    for i, f in enumerate(numerators):
        if f == 0:
            continue

        # 先检查是否真的只含 t 变量
        try:
            vars_f = f.variables()
        except Exception:
            vars_f = ()

        bad_vars = [v for v in vars_f if v not in t_vars]
        if len(bad_vars) != 0:
            failed.append((i, f, f"contains non-t vars: {bad_vars}"))
            continue

        try:
            g = f.subs(sub_map)
        except Exception:
            g = f

        try:
            g = Rsmall(g)
        except Exception as e:
            failed.append((i, f, f"coercion failed: {e}"))
            continue

        if g != 0:
            polys_small.append(g)

    if verbose:
        print("\nlen(polys_small) =", len(polys_small))
        for i, g in enumerate(polys_small[:10]):
            print(f"[{i}] {g}")
            print("parent =", g.parent())

        if len(failed) > 0:
            print("\nSkipped numerators:")
            for item in failed[:10]:
                idx, f, reason = item
                print(f"[{idx}] reason = {reason}")
                print("    ", f)

    if len(polys_small) == 0:
        return {
            "ring": Rsmall,
            "polys": [],
            "ideal": Rsmall.ideal([]),
            "groebner_basis": [],
            "dimension": None,
            "t_vars": t_vars,
            "failed": failed,
            "error": "no valid t-only numerators could be moved into Rsmall",
        }

    # 去重，避免重复方程拖慢
    polys_small = list(dict.fromkeys(polys_small))

    if verbose:
        print("\nafter dedup, len(polys_small) =", len(polys_small))

    J = Rsmall.ideal(polys_small)

    if verbose:
        print("\nIdeal J created successfully.")
        print(J)

    # 5) Groebner
    try:
        gb = J.groebner_basis()
    except Exception as e:
        if verbose:
            print("\nGroebner failed:")
            print(e)
        return {
            "ring": Rsmall,
            "polys": polys_small,
            "ideal": J,
            "groebner_basis": None,
            "dimension": None,
            "t_vars": t_vars,
            "failed": failed,
            "error": str(e),
        }

    # 6) 维数
    try:
        dimJ = J.dimension()
    except Exception as e:
        dimJ = None
        if verbose:
            print("\nDimension computation failed:")
            print(e)

    if verbose:
        print("\nGroebner basis:")
        for g in gb:
            print(g)
        print("dimension =", dimJ)

    return {
        "ring": Rsmall,
        "polys": polys_small,
        "ideal": J,
        "groebner_basis": gb,
        "dimension": dimJ,
        "t_vars": t_vars,
        "failed": failed,
    }

def solve_zero_dim_t_ideal(analysis, verbose=True):
    """
    若 analysis 对应的理想是 0 维，尝试直接求解。
    """
    J = analysis["ideal"]
    if J is None:
        print("理想为空。")
        return None

    if analysis["is_zero_dimensional"] is not True:
        print("这个理想目前看起来不是 0 维，或者还无法确认 0 维，不能直接按有限解集处理。")
        return None

    R = analysis["polys"][0].parent()

    # 方法1：尝试 variety
    try:
        sols = J.variety()
        print("variety() 找到的解：")
        for k, sol in enumerate(sols):
            print(f"solution {k}: {sol}")
        return sols
    except Exception as e:
        if verbose:
            print("J.variety() 失败：", e)

    # 方法2：如果基域不是代数闭域，尝试到分式域/QQbar
    try:
        from sage.all import QQbar
        S = PolynomialRing(QQbar, R.variable_names(), order='lex')
        gens2 = [S(f) for f in J.gens()]
        J2 = S.ideal(gens2)
        sols = J2.variety()
        print("在 QQbar 上 variety() 找到的解：")
        for k, sol in enumerate(sols):
            print(f"solution {k}: {sol}")
        return sols
    except Exception as e:
        if verbose:
            print("转到 QQbar 后 variety() 仍失败：", e)

    print("没能直接解出有限解集。")
    return None

def analyze_and_try_solve_t_conditions(info, verbose=True):
    analysis = analyze_t_ideal_from_info(info, verbose=verbose)

    if analysis["ideal"] is None:
        return analysis, None

    sols = None
    if analysis["is_zero_dimensional"] is True:
        sols = solve_zero_dim_t_ideal(analysis, verbose=verbose)
    else:
        print("暂时看不是 0 维，或者无法确认是 0 维。")
        print("这不一定表示无解，只是说明解集可能是正维的，或者 Groebner 分析还不够。")

    return analysis, sols

def simplify_polys_for_ideal(polys):
    """
    对多项式做一点简单清理：
    - 去掉 0
    - 尽量 primitive part / squarefree part
    - 去重
    """
    out = []
    seen = set()

    for f in polys:
        if f == 0:
            continue

        g = f
        try:
            # 如果有分数内容，先 primitive_part
            g = g.primitive_part()
        except Exception:
            pass

        try:
            # 平方自由部分
            g = g.squarefree_part()
        except Exception:
            pass

        s = str(g)
        if s not in seen and g != 0:
            seen.add(s)
            out.append(g)

    return out

def evaluate_expr_at_point(expr, pt):
    """
    在点 pt 处对 expr 求值。
    pt 形如 {"t0": a, "t1": b, "t2": c}
    """
    try:
        P = expr.parent()
        gens = P.gens()
    except Exception:
        return expr

    if len(gens) == 0:
        return expr

    vals = []
    used_any = False
    for g in gens:
        name = str(g)
        if name in pt:
            vals.append(pt[name])
            used_any = True
        else:
            vals.append(g)

    if not used_any:
        return expr

    try:
        return expr(*vals)
    except TypeError:
        return expr


def denominator_part(expr):
    """
    返回 expr 的分母；如果 expr 不是分式，就返回 1
    """
    try:
        return expr.denominator()
    except AttributeError:
        return 1


def collect_avoid_polynomial_from_info(info):
    """
    把当前 chart 下所有必须避开的分母收集起来，乘成一个 avoid_poly。
    只要 avoid_poly(pt) != 0，说明当前这套 c(t)/residual(t) 表达式在 pt 处合法。
    """
    polys = []

    # c_sol 的分母
    for ci in info["c_sol"]:
        d = denominator_part(ci)
        if d != 1:
            polys.append(d)

    # residuals 的分母
    for rj in info["residuals"]:
        d = denominator_part(rj)
        if d != 1:
            polys.append(d)

    if len(polys) == 0:
        return 1

    R = polys[0].parent()
    avoid = R.one()

    # 去重后相乘
    seen = set()
    for d in polys:
        s = str(d)
        if s not in seen:
            seen.add(s)
            avoid *= d

    return avoid


def specialize_matrix_at_point(M, pt):
    """
    把矩阵 M 在点 pt 处代值。
    若某项分母为 0，则报错。
    """
    rows = []

    for i in range(M.nrows()):
        row = []
        for j in range(M.ncols()):
            f = M[i, j]

            try:
                num = f.numerator()
                den = f.denominator()
            except AttributeError:
                num = f
                den = 1

            numv = evaluate_expr_at_point(num, pt)
            denv = evaluate_expr_at_point(den, pt)

            if denv == 0:
                raise ZeroDivisionError(
                    f"entry ({i},{j}) has denominator 0 at point {pt}"
                )

            row.append(numv / denv)

        rows.append(row)

    return matrix(rows)


def denominator_zero_positions(M, pt):
    """
    检查矩阵 M 在点 pt 处哪些位置分母为 0。
    返回 [(i, j, numv, denv), ...]
    """
    bad = []

    for i in range(M.nrows()):
        for j in range(M.ncols()):
            f = M[i, j]

            try:
                num = f.numerator()
                den = f.denominator()
            except AttributeError:
                num = f
                den = 1

            numv = evaluate_expr_at_point(num, pt)
            denv = evaluate_expr_at_point(den, pt)

            if denv == 0:
                bad.append((i, j, numv, denv))

    return bad


def last_row_in_span_after_specialize(M, pt, verbose=True):
    """
    直接用 rank 检查：
    代值后最后一行是否在前面各行张成空间里
    """
    M0 = specialize_matrix_at_point(M, pt)
    r_all = M0.rank()
    r_prev = M0[:-1, :].rank()
    ok = (r_all == r_prev)

    if verbose:
        print("rank(M0) =", r_all)
        print("rank(prefix) =", r_prev)
        print("last row in span ? ->", ok)

    return M0, ok


def find_point_by_slicing_strict(small, avoid_poly, Mcheck=None, trials=150, verbose=True):
    """
    在有限域上找点 pt，使得：
      1. small["polys"] = 0
      2. avoid_poly(pt) != 0
      3. 若给了 Mcheck，则额外要求：
         - Mcheck 在 pt 处没有分母爆炸
         - 且代值后最后一行真的在前缀行空间里
    """
    R = small["ring"]
    F = R.base_ring()
    polys = small["polys"]
    vars_ = R.gens()

    if len(vars_) != 3:
        raise ValueError("目前这个函数只处理 3 个变量。")

    t0, t1, t2 = vars_

    tested_slices = 0
    for a in F:
        tested_slices += 1
        if tested_slices > trials:
            break

        polys2_raw = [f(t0=a) for f in polys]

        R2 = PolynomialRing(F, [str(t1), str(t2)], order='lex')
        u1, u2 = R2.gens()

        polys2 = []
        for f in polys2_raw:
            try:
                ff = f.subs({t1: u1, t2: u2})
                polys2.append(R2(ff))
            except Exception:
                try:
                    polys2.append(R2(f))
                except Exception:
                    pass

        if len(polys2) == 0:
            continue

        J = R2.ideal(polys2)

        try:
            dimJ = J.dimension()
        except Exception:
            continue

        if dimJ != 0:
            continue

        try:
            sols = J.variety()
        except Exception:
            continue

        if not sols:
            continue

        for sol in sols:
            b = sol[u1] if u1 in sol else sol[str(u1)]
            c = sol[u2] if u2 in sol else sol[str(u2)]

            pt = {
                str(t0): a,
                str(t1): b,
                str(t2): c
            }

            # 必须避开所有分母
            if evaluate_expr_at_point(avoid_poly, pt) == 0:
                continue

            # 若给了原矩阵，再做最终真验证
            if Mcheck is not None:
                bad = denominator_zero_positions(Mcheck, pt)
                if len(bad) != 0:
                    continue

                _, ok = last_row_in_span_after_specialize(Mcheck, pt, verbose=False)
                if not ok:
                    continue

            if verbose:
                print("找到严格合法的点：", pt)
            return pt

    if verbose:
        print("没找到满足 numerators=0 且避开全部分母并通过最终验证的点。")
    return None

def find_point_by_slicing_no_chart(small, trials=120, verbose=True):
    """
    在有限域上找一个点：
    只要求满足 small["polys"] = 0
    不再检查任何 chart denominator。

    做法：
    固定 t0 = a 后，在二维里求 0 维切片；
    找到解就直接返回。
    """
    R = small["ring"]
    F = R.base_ring()
    polys = small["polys"]
    t0, t1, t2 = R.gens()

    vals = []
    k = 0
    for a in F:
        vals.append(a)
        k += 1
        if k >= trials:
            break

    for a in vals:
        polys2_raw = [f(t0=a) for f in polys]

        R2 = PolynomialRing(F, [str(t1), str(t2)], order='lex')
        u1, u2 = R2.gens()

        polys2 = []
        for f in polys2_raw:
            try:
                ff = f.subs({t1: u1, t2: u2})
                polys2.append(R2(ff))
            except Exception:
                try:
                    polys2.append(R2(f))
                except Exception:
                    pass

        if len(polys2) == 0:
            continue

        J = R2.ideal(polys2)

        try:
            dimJ = J.dimension()
        except Exception:
            continue

        if dimJ != 0:
            continue

        try:
            sols = J.variety()
        except Exception:
            continue

        if not sols:
            continue

        for sol in sols:
            b = sol[u1] if u1 in sol else sol[str(u1)]
            c = sol[u2] if u2 in sol else sol[str(u2)]

            pt = {
                str(t0): a,
                str(t1): b,
                str(t2): c
            }

            if verbose:
                print("找到满足 numerators=0 的候选点：", pt)
            return pt

    if verbose:
        print("在前", trials, "个切片里没找到候选点。")
    return None

def find_point_by_slicing_strict_debug(small, avoid_poly, Mcheck=None, trials=150, verbose=True):
    R = small["ring"]
    F = R.base_ring()
    polys = small["polys"]
    vars_ = R.gens()

    if len(vars_) != 3:
        raise ValueError("目前这个函数只处理 3 个变量。")

    t0, t1, t2 = vars_

    count_slice_total = 0
    count_zero_dim = 0
    count_candidates = 0
    count_pass_avoid = 0
    count_pass_den = 0
    count_pass_rank = 0

    for a in F:
        count_slice_total += 1
        if count_slice_total > trials:
            break

        polys2_raw = [f(t0=a) for f in polys]

        R2 = PolynomialRing(F, [str(t1), str(t2)], order='lex')
        u1, u2 = R2.gens()

        polys2 = []
        for f in polys2_raw:
            try:
                ff = f.subs({t1: u1, t2: u2})
                polys2.append(R2(ff))
            except Exception:
                try:
                    polys2.append(R2(f))
                except Exception:
                    pass

        if len(polys2) == 0:
            continue

        J = R2.ideal(polys2)

        try:
            dimJ = J.dimension()
        except Exception:
            continue

        if dimJ != 0:
            continue

        count_zero_dim += 1

        try:
            sols = J.variety()
        except Exception:
            continue

        if verbose:
            print(f"slice t0={a}: zero-dimensional, #sols={len(sols)}")

        for sol in sols:
            count_candidates += 1

            b = sol[u1] if u1 in sol else sol[str(u1)]
            c = sol[u2] if u2 in sol else sol[str(u2)]

            pt = {str(t0): a, str(t1): b, str(t2): c}

            av = evaluate_expr_at_point(avoid_poly, pt)
            if av == 0:
                if verbose:
                    print("  rejected by avoid_poly:", pt)
                continue

            count_pass_avoid += 1

            if Mcheck is not None:
                bad = denominator_zero_positions(Mcheck, pt)
                if len(bad) != 0:
                    if verbose:
                        print("  rejected by matrix denominator:", pt, "bad count =", len(bad))
                    continue

                count_pass_den += 1

                _, ok = last_row_in_span_after_specialize(Mcheck, pt, verbose=False)
                if not ok:
                    if verbose:
                        print("  rejected by final rank test:", pt)
                    continue

            count_pass_rank += 1
            print("找到严格合法的点：", pt)
            return pt

    print("\n===== debug summary =====")
    print("slices tested =", count_slice_total)
    print("zero-dimensional slices =", count_zero_dim)
    print("candidate points from varieties =", count_candidates)
    print("pass avoid_poly =", count_pass_avoid)
    print("pass matrix denominator =", count_pass_den)
    print("pass final rank =", count_pass_rank)

    return None

def test_candidate_on_original_matrix(M, pt, verbose=True):
    """
    不管 pt 是否落在 chart 分母上，
    直接代回原始矩阵 M，检查最后一行是否在前缀行空间里。
    """
    M0 = specialize_matrix_at_point(M, pt)

    r_all = M0.rank()
    r_prev = M0[:-1, :].rank()
    ok = (r_all == r_prev)

    if verbose:
        print("candidate pt =", pt)
        print("rank(M0) =", r_all)
        print("rank(prefix) =", r_prev)
        print("last row in span ? ->", ok)

    return M0, ok
def find_true_point_by_slicing_and_rank_check(small, Mcheck, trials=120, verbose=True):
    """
    先找满足 numerators=0 的候选点，
    再直接代回原始矩阵 Mcheck 检查 rank。
    不检查 chart denominator。
    """
    R = small["ring"]
    F = R.base_ring()
    polys = small["polys"]
    t0, t1, t2 = R.gens()

    vals = []
    k = 0
    for a in F:
        vals.append(a)
        k += 1
        if k >= trials:
            break

    for a in vals:
        polys2_raw = [f(t0=a) for f in polys]

        R2 = PolynomialRing(F, [str(t1), str(t2)], order='lex')
        u1, u2 = R2.gens()

        polys2 = []
        for f in polys2_raw:
            try:
                ff = f.subs({t1: u1, t2: u2})
                polys2.append(R2(ff))
            except Exception:
                try:
                    polys2.append(R2(f))
                except Exception:
                    pass

        if len(polys2) == 0:
            continue

        J = R2.ideal(polys2)

        try:
            dimJ = J.dimension()
        except Exception:
            continue

        if dimJ != 0:
            continue

        try:
            sols = J.variety()
        except Exception:
            continue

        if not sols:
            continue

        for sol in sols:
            b = sol[u1] if u1 in sol else sol[str(u1)]
            c = sol[u2] if u2 in sol else sol[str(u2)]

            pt = {
                str(t0): a,
                str(t1): b,
                str(t2): c
            }

            if verbose:
                print("testing candidate:", pt)

            try:
                M0 = specialize_matrix_at_point(Mcheck, pt)
            except Exception as e:
                if verbose:
                    print("  specialize failed:", e)
                continue

            ok = (M0.rank() == M0[:-1, :].rank())

            if verbose:
                print("  rank(M0) =", M0.rank())
                print("  rank(prefix) =", M0[:-1, :].rank())
                print("  ok =", ok)

            if ok:
                print("找到原始矩阵上的真解：", pt)
                return pt

    if verbose:
        print("没找到满足 numerators=0 且在原始矩阵上真正降 rank 的点。")
    return None

In [6]:
def _normalize_point_for_ring(pt, ring_or_gens):
    """
    把点 pt 统一成:
        { "t0": val0, "t1": val1, ... }
    支持:
        - dict: {"t0":..., "t1":...}
        - list/tuple: [..]
    """
    gens = ring_or_gens.gens() if hasattr(ring_or_gens, "gens") else tuple(ring_or_gens)

    if isinstance(pt, dict):
        out = {}
        for g in gens:
            name = str(g)
            if name in pt:
                out[name] = pt[name]
        return out

    if isinstance(pt, (list, tuple)):
        if len(pt) != len(gens):
            raise ValueError(f"需要 {len(gens)} 个参数值，但给了 {len(pt)} 个")
        return {str(gens[i]): pt[i] for i in range(len(gens))}

    raise TypeError("pt 必须是 dict / list / tuple")


def _values_from_point_for_ring(pt, ring_or_gens):
    """
    按 ring 的变量顺序返回代值列表。
    """
    gens = ring_or_gens.gens() if hasattr(ring_or_gens, "gens") else tuple(ring_or_gens)
    pt_dict = _normalize_point_for_ring(pt, gens)
    vals = []
    for g in gens:
        name = str(g)
        if name not in pt_dict:
            raise ValueError(f"点里缺少变量 {name} 的取值")
        vals.append(pt_dict[name])
    return vals


def _first_field_values(F, limit=None):
    """
    取有限域前 limit 个元素；limit=None 时取全域。
    """
    vals = []
    k = 0
    for a in F:
        vals.append(a)
        k += 1
        if limit is not None and k >= limit:
            break
    return vals


def _specialize_polys_fix_first_var(polys, vars_):
    """
    返回一个函数 specialize(a)，把 vars_[0]=a 代入后，
    把多项式列表重建到剩余变量的小环里。

    输出:
        rem_vars, specialize
    其中 specialize(a) 返回:
        - rem_vars 为空时: 常数/标量列表
        - rem_vars 非空时: 新环上的多项式列表
    """
    vars_ = list(vars_)
    v0 = vars_[0]
    rem_vars = vars_[1:]
    F = polys[0].parent().base_ring() if len(polys) > 0 else v0.parent().base_ring()

    if len(rem_vars) == 0:
        def specialize(a):
            out = []
            for f in polys:
                try:
                    out.append(f.subs({v0: a}))
                except Exception:
                    try:
                        out.append(f(v0=a))
                    except Exception:
                        out.append(f)
            return out
        return rem_vars, specialize

    R2 = PolynomialRing(F, [str(v) for v in rem_vars], order='lex')
    u = R2.gens()
    sub_map_rem = {rem_vars[i]: u[i] for i in range(len(rem_vars))}

    def specialize(a):
        out = []
        for f in polys:
            try:
                g = f.subs({v0: a})
            except Exception:
                try:
                    g = f(**{str(v0): a})
                except Exception:
                    g = f

            try:
                gg = g.subs(sub_map_rem)
                out.append(R2(gg))
            except Exception:
                try:
                    out.append(R2(g))
                except Exception:
                    pass
        return out

    return rem_vars, specialize


def _enumerate_points_by_recursive_slicing(polys, vars_, trials=150, verbose=False, prefix_pt=None):
    """
    通用递归切片：
      - 任意个 t 变量都能处理
      - 最后剩 0/1/2 个变量时直接解
      - 返回候选点 dict 的生成器

    注意：
      变量多时会变慢，trials 表示“每一层最多枚举多少个域元素”。
    """
    vars_ = list(vars_)
    prefix_pt = {} if prefix_pt is None else dict(prefix_pt)

    if len(vars_) == 0:
        ok = True
        for f in polys:
            try:
                if f != 0:
                    ok = False
                    break
            except Exception:
                if f != 0:
                    ok = False
                    break
        if ok:
            yield prefix_pt
        return

    # 只有一个变量：直接暴力验根
    if len(vars_) == 1:
        v = vars_[0]
        F = polys[0].parent().base_ring() if len(polys) > 0 else v.parent().base_ring()

        for a in _first_field_values(F, None):   # 1 维时直接全域扫
            good = True
            for f in polys:
                try:
                    val = f.subs({v: a})
                except Exception:
                    try:
                        val = f(v=a)
                    except Exception:
                        val = f
                if val != 0:
                    good = False
                    break
            if good:
                pt = dict(prefix_pt)
                pt[str(v)] = a
                yield pt
        return

    # 只剩两个变量：沿第一个变量切片，然后 variety()
    if len(vars_) == 2:
        v0, v1 = vars_
        F = polys[0].parent().base_ring() if len(polys) > 0 else v0.parent().base_ring()

        vals0 = _first_field_values(F, trials)
        for a in vals0:
            polys_raw = []
            for f in polys:
                try:
                    polys_raw.append(f.subs({v0: a}))
                except Exception:
                    try:
                        polys_raw.append(f(v0=a))
                    except Exception:
                        pass

            R2 = PolynomialRing(F, [str(v1)], order='lex')
            u1 = R2.gen()

            polys2 = []
            for f in polys_raw:
                try:
                    ff = f.subs({v1: u1})
                    polys2.append(R2(ff))
                except Exception:
                    try:
                        polys2.append(R2(f))
                    except Exception:
                        pass

            if len(polys2) == 0:
                continue

            J = R2.ideal(polys2)
            try:
                dimJ = J.dimension()
            except Exception:
                continue

            if dimJ != 0:
                # 维数不是 0，就退化成 1 维暴力
                for b in _first_field_values(F, None):
                    good = True
                    for f in polys2:
                        try:
                            if f(u1=b) != 0:
                                good = False
                                break
                        except Exception:
                            try:
                                if f.subs({u1: b}) != 0:
                                    good = False
                                    break
                            except Exception:
                                good = False
                                break
                    if good:
                        pt = dict(prefix_pt)
                        pt[str(v0)] = a
                        pt[str(v1)] = b
                        yield pt
                continue

            try:
                sols = J.variety()
            except Exception:
                continue

            for sol in sols:
                b = sol[u1] if u1 in sol else sol[str(u1)]
                pt = dict(prefix_pt)
                pt[str(v0)] = a
                pt[str(v1)] = b
                yield pt
        return

    # 变量 >= 3：继续递归切第一维
    F = polys[0].parent().base_ring()
    rem_vars, specialize = _specialize_polys_fix_first_var(polys, vars_)
    v0 = vars_[0]

    for a in _first_field_values(F, trials):
        polys2 = specialize(a)
        if len(polys2) == 0:
            continue

        pt2 = dict(prefix_pt)
        pt2[str(v0)] = a

        for sol in _enumerate_points_by_recursive_slicing(
            polys2,
            rem_vars,
            trials=trials,
            verbose=verbose,
            prefix_pt=pt2
        ):
            yield sol


def evaluate_expr_at_point(expr, pt):
    """
    在点 pt 处对 expr 求值。
    pt 可为:
        - {"t0": a, "t1": b, ...}
        - [a, b, ...] / (a, b, ...)
    """
    try:
        P = expr.parent()
        gens = P.gens()
    except Exception:
        return expr

    if len(gens) == 0:
        return expr

    pt_dict = _normalize_point_for_ring(pt, gens)

    vals = []
    used_any = False
    for g in gens:
        name = str(g)
        if name in pt_dict:
            vals.append(pt_dict[name])
            used_any = True
        else:
            vals.append(g)

    if not used_any:
        return expr

    try:
        return expr(*vals)
    except TypeError:
        try:
            return expr.subs({gens[i]: vals[i] for i in range(len(gens))})
        except Exception:
            return expr


def last_row_in_span_after_specialize(M, vals, verbose=True):
    """
    vals 可为:
        - 按顺序的 list/tuple
        - dict, 例如 {"t0":..., "t1":..., ...}

    返回代值后最后一行是否在前面行张成空间里
    """
    S = M.base_ring()
    F = S.base_ring()
    ts = S.gens()

    vals_list = _values_from_point_for_ring(vals, ts)
    sub_dict = {ts[i]: vals_list[i] for i in range(len(ts))}
    M0 = matrix(F, M.substitute(sub_dict))

    r_all = M0.rank()
    r_prev = M0[:-1].rank()

    if verbose:
        print("代入值 =", vals_list)
        print("rank(M0) =", r_all)
        print("rank(M0[:-1]) =", r_prev)
        print("最后一行是否在前面行空间里：", r_all == r_prev)

    return M0, (r_all == r_prev)


def find_point_by_slicing_strict(small, avoid_poly, Mcheck=None, trials=150, verbose=True):
    """
    在有限域上找点 pt，使得：
      1. small["polys"] = 0
      2. avoid_poly(pt) != 0
      3. 若给了 Mcheck，则额外要求：
         - Mcheck 在 pt 处没有分母爆炸
         - 且代值后最后一行真的在前缀行空间里

    现在支持任意个 t 变量。
    """
    R = small["ring"]
    polys = small["polys"]
    vars_ = list(R.gens())

    for pt in _enumerate_points_by_recursive_slicing(polys, vars_, trials=trials, verbose=verbose):
        if evaluate_expr_at_point(avoid_poly, pt) == 0:
            continue

        if Mcheck is not None:
            bad = denominator_zero_positions(Mcheck, pt)
            if len(bad) != 0:
                continue

            _, ok = last_row_in_span_after_specialize(Mcheck, pt, verbose=False)
            if not ok:
                continue

        if verbose:
            print("找到严格合法的点：", pt)
        return pt

    if verbose:
        print("没找到满足 numerators=0 且避开全部分母并通过最终验证的点。")
    return None


def find_point_by_slicing_no_chart(small, trials=120, verbose=True):
    """
    在有限域上找一个点：
    只要求满足 small["polys"] = 0
    不检查 chart denominator。
    现在支持任意个 t 变量。
    """
    R = small["ring"]
    polys = small["polys"]
    vars_ = list(R.gens())

    for pt in _enumerate_points_by_recursive_slicing(polys, vars_, trials=trials, verbose=verbose):
        if verbose:
            print("找到满足 numerators=0 的候选点：", pt)
        return pt

    if verbose:
        print("在递归切片搜索里没找到候选点。")
    return None


def find_point_by_slicing_strict_debug(small, avoid_poly, Mcheck=None, trials=150, verbose=True):
    """
    debug 版：支持任意个 t 变量。
    """
    R = small["ring"]
    polys = small["polys"]
    vars_ = list(R.gens())

    count_candidates = 0
    count_pass_avoid = 0
    count_pass_den = 0
    count_pass_rank = 0

    for pt in _enumerate_points_by_recursive_slicing(polys, vars_, trials=trials, verbose=verbose):
        count_candidates += 1

        av = evaluate_expr_at_point(avoid_poly, pt)
        if av == 0:
            if verbose:
                print("rejected by avoid_poly:", pt)
            continue

        count_pass_avoid += 1

        if Mcheck is not None:
            bad = denominator_zero_positions(Mcheck, pt)
            if len(bad) != 0:
                if verbose:
                    print("rejected by matrix denominator:", pt, "bad count =", len(bad))
                continue

            count_pass_den += 1

            _, ok = last_row_in_span_after_specialize(Mcheck, pt, verbose=False)
            if not ok:
                if verbose:
                    print("rejected by final rank test:", pt)
                continue

        count_pass_rank += 1
        print("找到严格合法的点：", pt)
        return pt

    print("\n===== debug summary =====")
    print("candidate points from recursive slicing =", count_candidates)
    print("pass avoid_poly =", count_pass_avoid)
    print("pass matrix denominator =", count_pass_den)
    print("pass final rank =", count_pass_rank)

    return None


def test_candidate_on_original_matrix(M, pt, verbose=True):
    """
    不管 pt 是否落在 chart 分母上，
    直接代回原始矩阵 M，检查最后一行是否在前缀行空间里。
    pt 可为 dict / list / tuple
    """
    M0 = specialize_matrix_at_point(M, pt)

    r_all = M0.rank()
    r_prev = M0[:-1, :].rank()
    ok = (r_all == r_prev)

    if verbose:
        print("candidate pt =", _normalize_point_for_ring(pt, M.base_ring()))
        print("rank(M0) =", r_all)
        print("rank(prefix) =", r_prev)
        print("last row in span ? ->", ok)

    return M0, ok


def find_true_point_by_slicing_and_rank_check(small, Mcheck, trials=120, verbose=True):
    """
    先找满足 numerators=0 的候选点，
    再直接代回原始矩阵 Mcheck 检查 rank。
    不检查 chart denominator。
    现在支持任意个 t 变量。
    """
    R = small["ring"]
    polys = small["polys"]
    vars_ = list(R.gens())

    for pt in _enumerate_points_by_recursive_slicing(polys, vars_, trials=trials, verbose=verbose):
        if verbose:
            print("testing candidate:", pt)

        try:
            M0 = specialize_matrix_at_point(Mcheck, pt)
        except Exception as e:
            if verbose:
                print("  specialize failed:", e)
            continue

        ok = (M0.rank() == M0[:-1, :].rank())

        if verbose:
            print("  rank(M0) =", M0.rank())
            print("  rank(prefix) =", M0[:-1, :].rank())
            print("  ok =", ok)

        if ok:
            print("找到原始矩阵上的真解：", pt)
            return pt

    if verbose:
        print("没找到满足 numerators=0 且在原始矩阵上真正降 rank 的点。")
    return None

def analyze_t_ideal_with_avoid(small, avoid_poly, verbose=True, compute_variety=False):
    """
    在扩环 F[t..., s] 中分析:
        small["polys"] = 0
        s * avoid_poly - 1 = 0

    这等价于只保留 avoid_poly != 0 的解。
    自动处理 avoid_poly 与 small["ring"] parent 不一致的问题。
    """

    R = small["ring"]
    polys = [f for f in small["polys"] if f != 0]
    t_vars = list(R.gens())
    F = R.base_ring()

    # 扩一个新变量 s
    Rext = PolynomialRing(F, [str(v) for v in t_vars] + ["s"], order='lex')
    gens_ext = Rext.gens()
    u_t = gens_ext[:-1]
    s = gens_ext[-1]

    # 先搬 small["polys"]
    sub_map_small = {t_vars[i]: u_t[i] for i in range(len(t_vars))}

    polys_ext = []
    for f in polys:
        g = f.subs(sub_map_small)
        polys_ext.append(Rext(g))

    # 再单独搬 avoid_poly：必须按 avoid_poly 自己的 parent 来替换
    Pavoid = avoid_poly.parent()
    avoid_gens = list(Pavoid.gens())

    name_to_ut = {str(t_vars[i]): u_t[i] for i in range(len(t_vars))}
    avoid_sub_map = {}

    for v in avoid_gens:
        name = str(v)
        if name in name_to_ut:
            avoid_sub_map[v] = name_to_ut[name]

    if verbose:
        print("avoid_poly parent =", Pavoid)
        print("small ring =", R)
        print("Rext =", Rext)
        print("avoid_sub_map =", avoid_sub_map)

    avoid_tmp = avoid_poly.subs(avoid_sub_map)
    avoid_ext = Rext(avoid_tmp)

    polys_ext.append(s * avoid_ext - 1)

    J = Rext.ideal(polys_ext)

    if verbose:
        print("number of generators =", len(polys_ext))
        print("last generator =")
        print(polys_ext[-1])

    try:
        is_one = J.is_one()
    except Exception:
        is_one = False

    if verbose:
        print("J.is_one() =", is_one)

    gb = None
    dimJ = None
    sols = None

    if not is_one:
        try:
            gb = J.groebner_basis()
            if verbose:
                print("\nGroebner basis computed.")
                if len(gb) <= 20:
                    for g in gb:
                        print(g)
                else:
                    print("GB size =", len(gb))
        except Exception as e:
            if verbose:
                print("\nGroebner basis failed:")
                print(e)

        try:
            dimJ = J.dimension()
            if verbose:
                print("dimension =", dimJ)
        except Exception as e:
            if verbose:
                print("dimension() failed:", e)

        if compute_variety and dimJ == 0:
            try:
                sols = J.variety()
                if verbose:
                    print("number of solutions =", len(sols))
                    if len(sols) <= 10:
                        for sol in sols:
                            print(sol)
            except Exception as e:
                if verbose:
                    print("variety() failed:", e)

    return {
        "ring": Rext,
        "ideal": J,
        "groebner_basis": gb,
        "dimension": dimJ,
        "is_one": is_one,
        "solutions": sols,
        "generators": polys_ext,
        "avoid_ext": avoid_ext,
    }

In [7]:
import random

def _random_field_element(F):
    try:
        return F.random_element()
    except Exception:
        vals = list(F)
        return random.choice(vals)

def _substitute_by_name(expr, target_ring):
    P = expr.parent()
    src_gens = list(P.gens())
    tgt_gens = list(target_ring.gens())
    name_to_tgt = {str(g): g for g in tgt_gens}

    sub_map = {}
    for v in src_gens:
        sv = str(v)
        if sv in name_to_tgt:
            sub_map[v] = name_to_tgt[sv]

    try:
        out = expr.subs(sub_map)
    except Exception:
        out = expr

    return target_ring(out)

def _specialize_expr_by_name(expr, assignments):
    P = expr.parent()
    src_gens = list(P.gens())

    sub_map = {}
    for v in src_gens:
        sv = str(v)
        if sv in assignments:
            sub_map[v] = assignments[sv]

    try:
        return expr.subs(sub_map)
    except Exception:
        return expr

def _build_slice_ring_and_system(small, avoid_poly, fixed_assignments, verbose=False):
    R = small["ring"]
    polys = [f for f in small["polys"] if f != 0]
    all_t = list(R.gens())
    F = R.base_ring()

    remaining_t = [t for t in all_t if str(t) not in fixed_assignments]

    polys_spec_raw = []
    for f in polys:
        g = _specialize_expr_by_name(f, fixed_assignments)
        if g != 0:
            polys_spec_raw.append(g)

    avoid_spec_raw = _specialize_expr_by_name(avoid_poly, fixed_assignments)

    Rslice = PolynomialRing(F, [str(v) for v in remaining_t] + ["s"], order='lex')
    gens_slice = list(Rslice.gens())
    s = gens_slice[-1]

    polys_slice = []
    for g in polys_spec_raw:
        gg = _substitute_by_name(g, Rslice)
        if gg != 0:
            polys_slice.append(gg)

    avoid_slice = _substitute_by_name(avoid_spec_raw, Rslice)
    polys_slice.append(s * avoid_slice - 1)

    if verbose:
        print("fixed_assignments =", fixed_assignments)
        print("remaining_t =", remaining_t)
        print("Rslice =", Rslice)
        print("num generators =", len(polys_slice))
        print("last generator =", polys_slice[-1])

    return {
        "ring": Rslice,
        "generators": polys_slice,
        "remaining_t": remaining_t,
        "fixed_assignments": fixed_assignments,
        "avoid_slice": avoid_slice,
    }

def _analyze_slice_system(slice_data, compute_gb=True, compute_variety_if_zero_dim=True, verbose=False):
    Rslice = slice_data["ring"]
    gens = slice_data["generators"]
    J = Rslice.ideal(gens)

    result = {
        "ideal": J,
        "classification": None,
        "dimension": None,
        "is_one": None,
        "gb": None,
        "solutions": None,
        "error": None,
    }

    try:
        is_one = J.is_one()
    except Exception as e:
        is_one = None
        result["error"] = f"is_one failed: {e}"

    result["is_one"] = is_one

    if is_one is True:
        result["classification"] = "one_ideal"
        return result

    gb = None
    if compute_gb:
        try:
            gb = J.groebner_basis()
            result["gb"] = gb
        except Exception as e:
            result["classification"] = "gb_failed"
            result["error"] = f"groebner_basis failed: {e}"
            return result

    try:
        dimJ = J.dimension()
        result["dimension"] = dimJ
    except Exception as e:
        result["classification"] = "gb_failed"
        old = result["error"]
        result["error"] = (old + " ; " if old else "") + f"dimension failed: {e}"
        return result

    if dimJ == 0:
        if compute_variety_if_zero_dim:
            try:
                sols = J.variety()
                result["solutions"] = sols
                if len(sols) == 0:
                    result["classification"] = "zero_dim_no_solution"
                else:
                    result["classification"] = "zero_dim_has_solution"
            except Exception as e:
                result["classification"] = "variety_failed"
                old = result["error"]
                result["error"] = (old + " ; " if old else "") + f"variety failed: {e}"
        else:
            result["classification"] = "zero_dim"
    else:
        result["classification"] = "positive_dim"

    return result

def _merge_full_point(all_t, fixed_assignments, sol):
    """
    把切片解 sol 与 fixed_assignments 合并成原始 ring 上完整的 t 点。
    自动去掉 s。
    返回:
        {"t0":..., "t1":..., ...}
    """
    full = dict(fixed_assignments)

    for k, v in sol.items():
        sk = str(k)
        if sk == "s":
            continue
        full[sk] = v

    # 按 all_t 顺序整理
    ordered = {}
    for t in all_t:
        st = str(t)
        if st in full:
            ordered[st] = full[st]
    return ordered

def random_slice_diagnose(
    small,
    avoid_poly,
    num_fix=4,
    num_trials=20,
    seed=None,
    compute_gb=True,
    compute_variety_if_zero_dim=True,
    verbose_each=False,
    stop_on_first_good=False,
):
    """
    随机固定 num_fix 个 t，分析低维切片。
    现在会自动保存找到的完整参数点:
        summary["good_points"]
        summary["first_good_point"]
    """
    if seed is not None:
        random.seed(seed)

    R = small["ring"]
    all_t = list(R.gens())
    F = R.base_ring()

    if num_fix < 0 or num_fix > len(all_t):
        raise ValueError(f"num_fix 必须在 0 到 {len(all_t)} 之间")

    summary = {
        "num_trials": num_trials,
        "num_fix": num_fix,
        "all_t": all_t,
        "counts": {
            "one_ideal": 0,
            "zero_dim_no_solution": 0,
            "zero_dim_has_solution": 0,
            "positive_dim": 0,
            "gb_failed": 0,
            "variety_failed": 0,
            "zero_dim": 0,
            "other": 0,
        },
        "trials": [],
        "good_points": [],
        "first_good_point": None,
    }

    for trial_id in range(num_trials):
        chosen = random.sample(all_t, num_fix)
        fixed_assignments = {str(t): _random_field_element(F) for t in chosen}

        try:
            slice_data = _build_slice_ring_and_system(
                small,
                avoid_poly,
                fixed_assignments,
                verbose=verbose_each
            )
            analysis = _analyze_slice_system(
                slice_data,
                compute_gb=compute_gb,
                compute_variety_if_zero_dim=compute_variety_if_zero_dim,
                verbose=verbose_each
            )
        except Exception as e:
            slice_data = None
            analysis = {
                "classification": "other",
                "dimension": None,
                "is_one": None,
                "gb": None,
                "solutions": None,
                "error": str(e),
            }

        full_points = []
        if analysis.get("classification") == "zero_dim_has_solution" and analysis.get("solutions") is not None:
            for sol in analysis["solutions"]:
                full_pt = _merge_full_point(all_t, fixed_assignments, sol)
                full_points.append(full_pt)
                summary["good_points"].append(full_pt)
                if summary["first_good_point"] is None:
                    summary["first_good_point"] = full_pt

        record = {
            "trial_id": trial_id,
            "fixed_assignments": fixed_assignments,
            "remaining_t": None if slice_data is None else slice_data["remaining_t"],
            "classification": analysis["classification"],
            "dimension": analysis.get("dimension", None),
            "is_one": analysis.get("is_one", None),
            "num_solutions": None if analysis.get("solutions", None) is None else len(analysis["solutions"]),
            "solutions": analysis.get("solutions", None),
            "full_points": full_points,
            "error": analysis.get("error", None),
        }

        cls = record["classification"]
        if cls in summary["counts"]:
            summary["counts"][cls] += 1
        else:
            summary["counts"]["other"] += 1

        summary["trials"].append(record)

        msg = f"[trial {trial_id}] class={cls}, dim={record['dimension']}, fixed={fixed_assignments}"
        if len(full_points) > 0:
            msg += f"  <-- found {len(full_points)} full point(s)"
        print(msg)

        if stop_on_first_good and cls == "zero_dim_has_solution":
            print("发现 zero_dim_has_solution，提前停止。")
            break

    print("\n===== summary =====")
    for k, v in summary["counts"].items():
        if v:
            print(f"{k}: {v}")

    print("total good full points saved =", len(summary["good_points"]))
    if summary["first_good_point"] is not None:
        print("first_good_point =", summary["first_good_point"])

    return summary

def print_good_trials(summary, max_show=10):
    good_classes = {"zero_dim_has_solution", "zero_dim_no_solution", "one_ideal"}
    shown = 0
    for rec in summary["trials"]:
        if rec["classification"] in good_classes:
            print("\n------------------------------")
            print("trial_id =", rec["trial_id"])
            print("classification =", rec["classification"])
            print("dimension =", rec["dimension"])
            print("fixed_assignments =", rec["fixed_assignments"])
            print("remaining_t =", rec["remaining_t"])
            print("num_solutions =", rec["num_solutions"])
            if rec["solutions"] is not None and rec["num_solutions"] is not None and rec["num_solutions"] <= 5:
                print("solutions =", rec["solutions"])
            if rec["full_points"] is not None and len(rec["full_points"]) > 0:
                print("full_points =", rec["full_points"])
            if rec["error"] is not None:
                print("error =", rec["error"])
            shown += 1
            if shown >= max_show:
                break

def restrict_f3_to_line_via_x2(f1, f2, f3, x0, x1, x2, verbose=True):
    """
    输入:
        f1, f2, f3 : Sage 多项式
        x0, x1, x2 : 你想消去/保留的三个变量
                     这里默认把 x2 当作交线参数 z
    输出:
        一个字典，包含:
          - det      : 解 x0,x1 时的分母
          - x0_on_L  : 交线上 x0 的表达式
          - x1_on_L  : 交线上 x1 的表达式
          - g        : f3 限制到 V(f1,f2) 后关于 x2 的多项式
          - coeffs   : {0:c0, 1:c1, 2:c2, 3:c3}
    """

    R = f1.parent()
    F = R.fraction_field()

    # 提升到分式域，避免除法问题
    f1F = F(f1)
    f2F = F(f2)
    f3F = F(f3)

    # 提取关于 x0,x1,x2 的一次系数与常数项
    def affine_parts(f):
        c0 = f.subs({x0:0, x1:0, x2:0})
        a0 = f.coefficient(x0)
        a1 = f.coefficient(x1)
        a2 = f.coefficient(x2)
        # 检查确实是仿射线性
        rem = f - (a0*x0 + a1*x1 + a2*x2 + c0)
        if rem != 0:
            print("Warning: 这个多项式不是关于 x0,x1,x2 的仿射一次式:")
            print(rem)
        return a0, a1, a2, c0

    # a00, a01, a02, b0 = affine_parts(f1F)
    # a10, a11, a12, b1 = affine_parts(f2F)
    a00, a01, a02, b0 = affine_parts(f1)
    a10, a11, a12, b1 = affine_parts(f2)

    # 解方程:
    # a00*x0 + a01*x1 + a02*x2 + b0 = 0
    # a10*x0 + a11*x1 + a12*x2 + b1 = 0
    #
    # 把 x2 视为自由变量，解 x0,x1
    det = a00*a11 - a01*a10

    if det == 0:
        raise ValueError("用于解 x0,x1 的 2x2 子式恒等为 0；需要换一个自由变量或换一个子式。")

    # rhs0 = -(a02*x2 + b0)
    # rhs1 = -(a12*x2 + b1)

    # x0_on_L = ( rhs0*a11 - a01*rhs1 ) / det
    # x1_on_L = ( a00*rhs1 - rhs0*a10 ) / det

    # # 代回 f3
    # g = f3F.subs({
    #     x0: x0_on_L,
    #     x1: x1_on_L
    # }).expand()
    rhs0 = F(-(a02*x2 + b0))
    rhs1 = F(-(a12*x2 + b1))
    det  = F(det)

    x0_on_L = (rhs0*a11 - a01*rhs1) / det
    x1_on_L = (a00*rhs1 - rhs0*a10) / det

    g = F(f3).subs({
        x0: x0_on_L,
        x1: x1_on_L
    })
    # 这时 g 应该只依赖 x2（以及 t 参数）
    # 抽取 x2^0, x2^1, x2^2, x2^3 的系数
    # coeffs = {}
    # for k in range(4):
    #     coeffs[k] = g.coefficient(x2, k)
    num = g.numerator()
    den = g.denominator()
    coeffs = {}
    for k in range(4):
        coeffs[k] = num.coefficient(x2^k)

    if verbose:
        print("="*70)
        print("det =")
        print(det)
        print("="*70)
        print("x0 on V(f1,f2) =")
        print(x0_on_L)
        print("="*70)
        print("x1 on V(f1,f2) =")
        print(x1_on_L)
        print("="*70)
        print("g(x2) = f3|_{V(f1,f2)} =")
        print(g)
        print("="*70)
        print("coefficients of 1, x2, x2^2, x2^3:")
        for k in range(4):
            print(f"coeff x2^{k} =")
            print(coeffs[k])
            print("-"*50)

    return {
        "det": det,
        "x0_on_L": x0_on_L,
        "x1_on_L": x1_on_L,
        "g": g,
        "num": num,
        "den": den,
        "coeffs": coeffs,
    }

from itertools import product

def search_point_via_left_kernel_slice(M, fixed_pt, free_vars, value_pool, verbose=True):
    """
    M: 原矩阵，比如 Mcrit 或 Ms[0]
    fixed_pt: dict，例如 {"t3":1, "t4":7, ...}，固定大部分 t
    free_vars: 剩下自由变量，例如 [t0, t1, t2]
    value_pool: 搜索取值，例如 range(20) 或 finite field 元素列表

    返回:
        {
            "kernel_dim": ...,
            "good_basis_vectors": [...],   # 最后一个坐标不恒为0的左核基向量
            "point": {...} or None         # 找到的完整 t 点
        }
    """

    # 1. 先固定大部分 t
    M_sub = M.substitute(fixed_pt)

    # 2. 在分式域上算左核
    S = M_sub.base_ring()
    Kfrac = S.fraction_field()
    Mf = matrix(Kfrac, M_sub)

    K = Mf.left_kernel()
    basis = K.basis()

    if verbose:
        print("left kernel dim =", K.dimension())
        print("basis size =", len(basis))

    # 3. 只保留“最后一个坐标不恒等于0”的左核向量
    good_basis = []
    for v in basis:
        if v[-1] != 0:
            good_basis.append(v)

    if verbose:
        print("basis vectors with nonzero last entry =", len(good_basis))
        for i, v in enumerate(good_basis[:5]):
            print(f"\n[good basis {i}]")
            print(v)
            print("last entry =", v[-1])

    if len(good_basis) == 0:
        return {
            "kernel_dim": K.dimension(),
            "good_basis_vectors": [],
            "point": None
        }

    # 4. 搜自由变量取值
    for vals in product(value_pool, repeat=len(free_vars)):
        pt = dict(fixed_pt)
        for x, a in zip(free_vars, vals):
            pt[str(x)] = a

        # 检查是否有某个左核向量在该点合法且最后坐标非0
        for v in good_basis:
            ok = True
            eval_v = []

            for entry in v:
                den = denominator_part(entry)
                denv = evaluate_expr_at_point(den, pt)
                if denv == 0:
                    ok = False
                    break
                eval_v.append(evaluate_expr_at_point(entry, pt))

            if not ok:
                continue

            if eval_v[-1] == 0:
                continue

            # 5. 最终直接检查 rank
            try:
                M0, rank_ok = last_row_in_span_after_specialize(M, pt, verbose=False)
            except Exception:
                continue

            if rank_ok:
                if verbose:
                    print("\n找到点：", pt)
                    print("specialized kernel vector =", eval_v)
                    print("last entry =", eval_v[-1])
                    print("rank check = True")
                return {
                    "kernel_dim": K.dimension(),
                    "good_basis_vectors": good_basis,
                    "point": pt
                }

    if verbose:
        print("没找到满足条件的点。")

    return {
        "kernel_dim": K.dimension(),
        "good_basis_vectors": good_basis,
        "point": None
    }

### Part 2: Construction of the nonlinear subspace path

The second part implements the construction method used in Section 3.3. It
uses the available constraint budget \(E_c=12\) to construct a nonlinear
subspace path for the internal rounds of the \(t=16\) KoalaBear instance.

This part is the main constructive step of the notebook. It determines the
constraints that define the nonlinear path.

### Part 2.1: Poseidon class set up

In [ ]:
from sage.all import *
import random

class PoseidonSage:
    def __init__(self, t=16, rate=14, capacity=2, s=1, p=2130706433, alpha=3, n_rounds=30, seed=None):
        """####4294967311
        Poseidon toy model in Sage
        """
        assert rate + capacity == t, "rate + capacity must equal t"

        self.t = t
        self.rate = rate
        self.capacity = capacity
        self.s = s
        self.p = p
        self.alpha = alpha
        self.n_rounds = n_rounds
        self.seed = seed
        self.F = GF(p)

        if seed is not None:
            random.seed(seed)

        # round constants and MDS matrix
        self.round_constants = self._generate_round_constants()
        self.mds_matrix = self._generate_cauchy_mds()

    # ------------------ Round constants ------------------
    def _generate_round_constants(self):
        RC = []
        for _ in range(self.n_rounds):
            RC.append(vector(self.F, [self.F(random.randint(0, self.p-1)) for _ in range(self.t)]))
        return RC

    # ------------------ Cauchy MDS matrix ------------------
    def _generate_cauchy_mds(self):
        elems = random.sample(range(self.p), 2*self.t)
        x = elems[:self.t]
        y = elems[self.t:]
        M = matrix(self.F, self.t, self.t)
        for i in range(self.t):
            for j in range(self.t):
                diff = x[i] - y[j]
                assert diff != 0, "Zero denominator in Cauchy construction"
                M[i,j] = self.F(1)/self.F(diff)
        return M

    # ------------------ S-box ------------------
    def sbox(self, x):
        return x ** self.alpha

    def external_sbox_layer(self, state):
        return vector(self.F, [self.sbox(x) for x in state])

    def internal_sbox_layer(self, state):
        state = vector(state)
        for i in range(self.s):
            state[i] = self.sbox(state[i])
        return state

    def inv_sbox(self, x):
        alpha_inv = inverse_mod(self.alpha, self.p - 1)
        return x ** alpha_inv

    def inv_internal_sbox_layer(self, state):
        state = vector(state)
        for i in range(self.s):
            state[i] = self.inv_sbox(state[i])
        return state
    
    def inv_mds_layer(self, state):
        return self.mds_matrix.inverse() * state

    def sub_round_constants(self, state, round_idx):
        return state - self.round_constants[round_idx]
    # def sub_round_constants(self, state, round_idx):
    #     R = state[0].parent()
    #     rc = vector(R, [R(c) for c in self.round_constants[round_idx]])
    #     return state - rc

    def inv_internal_round(self, state, round_idx):
        state = self.inv_mds_layer(state)
        state = self.inv_internal_sbox_layer(state)
        state = self.sub_round_constants(state, round_idx)
        return state
    # ------------------ Linear layer ------------------
    def mds_layer(self, state):
        return self.mds_matrix * state

    # ------------------ Round functions ------------------
    def add_round_constants(self, state, round_idx):
        return state + self.round_constants[round_idx]

    def external_round(self, state, round_idx):
        state = self.add_round_constants(state, round_idx)
        state = self.external_sbox_layer(state)
        state = self.mds_layer(state)
        return state

    def internal_round(self, state, round_idx):
        state = self.add_round_constants(state, round_idx)
        state = self.internal_sbox_layer(state)
        state = self.mds_layer(state)
        return state

    # ------------------ Kernel / Subspace search ------------------
    def kernel_mod_p(self, M):
        """
        Compute kernel of M over GF(p)
        Return as a matrix whose columns are basis vectors
        """
        return M.right_kernel().matrix().transpose()

    def compute_mds_invariant_subspaces_sage(self):
        p = self.p
        s = self.s
        Fp = GF(p)

        # MDS matrix mod p
        M = Matrix(ZZ, self.mds_matrix).apply_map(lambda x: x % p).change_ring(Fp)

        B = M[:s, s:]
        D = M[s:, s:]

        D_inv = D.inverse()


        W = self.kernel_mod_p(B).change_ring(Fp)
        W_list = [W]

        while W.ncols() > 0:
            # W2 = D^{-1} W
            W2 = D_inv * W

            # Z = [ W | -W2 ]
            Z = W.augment(-W2)

            # 求核
            ns = self.kernel_mod_p(Z).change_ring(Fp)
            if ns.ncols() == 0:
                break


            n = W.nrows()
            k_old = W.ncols()


            new_vectors = []
            for i in range(ns.ncols()):
                coeffs = vector(Fp, [ns[j, i] for j in range(W.ncols())])
                v = W * coeffs          
                new_vectors.append(v)  

            if not new_vectors:
                break

  
            W = matrix(Fp, new_vectors).transpose()
            W_list.append(W)

        return W_list

    def show_invariant_subspace_dynamics(self, W_list):
        p = self.p
        s = self.s
        M = Matrix(ZZ, self.mds_matrix).apply_map(lambda x: x % p)
        n = M.nrows()

        for idx, W in enumerate(W_list, start=1):
            print(f"\n{'='*16} W_{idx} {'='*16}")
            print(f"Dimension: {W.ncols()}")
            print("Basis (columns):")
            print(W)


            Z = zero_matrix(ZZ, s, W.ncols())
            V = Z.stack(W).apply_map(lambda x: x % p)

            print("\nInitial vectors after padding zeros:")
            print(V)

  
            cur = V
            for step in range(1, idx + 1):
                cur = (M * cur).apply_map(lambda x: x % p)
                print(f"\nAfter multiplying M^{step}:")
                print(cur)


        # ------------------ CICO equations over GF(p) ------------------
    
    def make_symbolic_X(self, k):
        """

        """
        Fp = self.F
        R = PolynomialRing(Fp, [f"x{i}" for i in range(k)])
        X = vector(R, R.gens())
        return R, X
    
    def make_symbolic_X_with_t(self, k, num_t):
        Fp = self.F

        x_vars = [f"x{i}" for i in range(k)]
        t_vars = [f"t{i}" for i in range(num_t)]

        R = PolynomialRing(Fp, x_vars + t_vars)

        gens = R.gens()

        X = vector(R, gens[:k])
        T = vector(R, gens[k:k+num_t])

        return R, X, T

    def initial_state_from_W(self, W, X):
        """

        """
        R = X[0].parent()
        s = self.s

        lin_part = list(W.change_ring(R) * X)  
        state_list = [R.zero()] * s + lin_part

        return vector(R, state_list)
    
    def rp_round_polynomial(self, state, round_index):
        """

        """
        R = state[0].parent()
        Fp = self.F
        s  = self.s

    
        rc = vector(R, [R(c) for c in self.round_constants[round_index]])
        state = state + rc

        # internal S-box
        state = vector(R, state)
        for i in range(s):
            state[i] = state[i] ** self.alpha

        # MDS
        M = self.mds_matrix.change_ring(R)
        state = M * state

        return state
    def rf_round_polynomial(self, state, round_index):
        """
     
        """
        R = state[0].parent()

    
        rc = vector(R, [R(c) for c in self.round_constants[round_index]])
        state = state + rc

        # external S-box
        state = vector(R, [x ** self.alpha for x in state])

        # MDS
        M = self.mds_matrix.change_ring(R)
        state = M * state

        return state
    # def generate_cico_equations_modp(self, W, constants, rounds, round_type="internal"):
    def generate_cico_equations_modp(self, W, constants, rounds, exter1, inter, exter2):
        """
        round_type: "internal" or "external"
        """
        k = W.ncols()
        R, X = self.make_symbolic_X(k)

      
        state = self.initial_state_from_W(W, X)

        assert exter1 + inter + exter2 == rounds
        round_idx = 0

   
        for _ in range(exter1):
            state = self.rf_round_polynomial(state, round_idx)
            round_idx += 1
        for _ in range(inter):
            state = self.rp_round_polynomial(state, round_idx)
            round_idx += 1
        for _ in range(exter2):
            state = self.rf_round_polynomial(state, round_idx)
            round_idx += 1

        # for r in range(rounds):
        #     if round_type == "internal":
        #         state = self.rp_round_polynomial(state, r)
        #     else:
        #         state = self.rf_round_polynomial(state, r)

  
        const_vec = vector(R, [R(c) for c in constants])
        equations = list(state - const_vec)

        return equations, R, X

    def generate_cico_equations_withnosubspace(self, constants, rounds, exter1, inter, exter2):
        """
        round_type: "internal" or "external"
        """
        k = self.t
        R, X = self.make_symbolic_X(k)
        W = identity_matrix(self.F, self.t)
    
        state = W.change_ring(R) * X

        assert exter1 + inter + exter2 == rounds
        round_idx = 0

    
        for _ in range(exter1):
            state = self.rf_round_polynomial(state, round_idx)
            round_idx += 1
        for _ in range(inter):
            state = self.rp_round_polynomial(state, round_idx)
            round_idx += 1
        for _ in range(exter2):
            state = self.rf_round_polynomial(state, round_idx)
            round_idx += 1

        # for r in range(rounds):
        #     if round_type == "internal":
        #         state = self.rp_round_polynomial(state, r)
        #     else:
        #         state = self.rf_round_polynomial(state, r)

        const_vec = vector(R, [R(c) for c in constants])
        equations = list(state - const_vec)

        return equations, R, X

    def simple_form(self, target, rounds):
        state = target
        for round_idx in range(rounds):
            state = self.inv_internal_round(state, round_idx)
        return state
    
    def retain_dim_shear_nosubspace(self, constants, l):#####
        k = self.t - l #########
        num_t = 20######
        num_u =25
        # R, X, t= self.make_symbolic_X_with_t(k)
        R, X, T = self.make_symbolic_X_with_t(k, num_t)
        # ：u0, u1, ..., u_{num_u-1}
        R_old = R
        old_names = [str(g) for g in R_old.gens()]
        u_names = [f"u{i}" for i in range(num_u)]

        R = PolynomialRing(R_old.base_ring(), old_names + u_names, order='degrevlex')
        gens_new = R.gens()
        name_to_gen = {nm: gens_new[i] for i, nm in enumerate(old_names + u_names)}

    
        phi = R_old.hom([name_to_gen[str(g)] for g in R_old.gens()], R)

    
        X = [name_to_gen[str(x)] for x in X]
        T = [name_to_gen[str(t)] for t in T]
        U = [name_to_gen[f"u{i}"] for i in range(num_u)]


        if l == 2:
            W = self.compute_mds_invariant_subspaces_sage()[l-2]
            W_R = matrix(R, W.nrows(), W.ncols(), [R(a) for a in W.list()])

         
            subspace_coords = vector(R, [
                T[0]*X[0],
                T[1]*X[1],
                T[2]*X[2]
            ])

            tail_lin = W_R * subspace_coords

       
            tail = vector(R, [
                tail_lin[0] + T[4],
                tail_lin[1] + T[5],
                tail_lin[2] + T[6],
                tail_lin[3] + T[7],
            ])

            state = vector(R, [T[3]] + list(tail))

        elif l == 1:
            # state = vector(R, [T[19]*X[0]+T[0], 
            #                    T[20]*X[1]+T[1],
            #                    T[21]*X[2]+T[2],
            #                    T[22]*X[3]+T[3],
            #                    R.zero()])
            state = vector(R, [T[0], 
                                T[1]*X[0]+T[2],
                                T[3]*X[1]+T[4],
                                T[5]*X[2]+T[6],
                                R.zero()
                                ])
        elif l == 0:
            rc0_first = R(self.round_constants[0][0])###################KoalarBear prime,Ec=16-4=12,c=d=2
            state = vector(R, [
                                U[0] - rc0_first,
                                X[0],
                                X[1],
                                X[2],
                                X[3],
                                X[4],
                                X[5],
                                X[6],
                                X[7],
                                X[8],
                                X[9],
                                X[10],
                                X[11],   
                                X[12],
                                X[13],
                                X[14]+X[15]                           
                                ])
                
        round_idx = 0

     
        eqs = []
        reduced_ideal = []

      


        Ec=12
        for i in range(Ec):
            state = self.rp_round_polynomial(state, round_idx)
            round_idx += 1
            f = R(state[0])
            eq = f - (U[i+1] - R(self.round_constants[i+1][0]))#####################
            # eqs.append(eq)
            reduced_ideal.append(eq)
            state = vector(R, [U[i+1] - R(self.round_constants[i+1][0])] + list(state[1:]))
###############################################################################################################


        for i in range(Ec):
            state = self.rp_round_polynomial(state, round_idx)
            round_idx += 1
            f = R(state[0])
            eq = f - (U[i+Ec+1] - R(self.round_constants[i+Ec+1][0]))###########
            eqs.append(eq)
            # reduced_ideal.append(eq)
            state = vector(R, [U[i+Ec+1] - R(self.round_constants[i+Ec+1][0])] + list(state[1:]))

        return eqs, reduced_ideal, R, X, T
    
def square_submatrix(M):
    n = M.ncols()
    return M[:n, :]

poseidon = PoseidonSage(seed=0)
W_list = poseidon.compute_mds_invariant_subspaces_sage()
poseidon.show_invariant_subspace_dynamics(W_list)



================ W_1 ================
Dimension: 14
Basis (columns):
[         1          0          0          0          0          0          0          0          0          0          0          0          0          0]
[         0          1          0          0          0          0          0          0          0          0          0          0          0          0]
[         0          0          1          0          0          0          0          0          0          0          0          0          0          0]
[         0          0          0          1          0          0          0          0          0          0          0          0          0          0]
[         0          0          0          0          1          0          0          0          0          0          0          0          0          0]
[         0          0          0          0          0          1          0          0          0          0          0          0          0       

/tmp/ipykernel_968176/235238949.py:22: DeprecationWarning: Seeding based on hashing is deprecated
since Python 3.9 and will be removed in a subsequent version. The only 
supported seed types are: None, int, float, str, bytes, and bytearray.
  random.seed(seed)


### Part2.2: Construct nonlinear subspace constraints

In [16]:
constants = [0]*16
# constants = [9, 123, 5411, 524, 9871]
# specific_p = 4294967311
specific_p = 2130706433

eqs, reduced_ideal, R, X, T = poseidon.retain_dim_shear_nosubspace(constants, l=0)

print("reduced")
for i in range(len(reduced_ideal)):
    print(reduced_ideal[i])
print("eqs")
for i in range(len(eqs)):
    print(eqs[i])

reduced
632975430*u0^3 - 316378788*x0 - 115131691*x1 + 277567332*x2 + 1036634045*x3 - 982951905*x4 - 1022922865*x5 - 913170848*x6 - 163239652*x7 - 97521843*x8 + 538345662*x9 + 401662602*x10 - 790415880*x11 + 1010438582*x12 - 22199362*x13 - 312414866*x14 - 312414866*x15 - u1 - 955613587
688082254*u0^3 + 632975430*u1^3 - 787822595*x0 - 968691862*x1 + 244286655*x2 + 1062102678*x3 - 1045599377*x4 + 192515438*x5 + 364552943*x6 - 1030002769*x7 - 667562765*x8 + 256856916*x9 + 443192814*x10 + 492450764*x11 - 913147802*x12 + 947970907*x13 - 977798047*x14 - 977798047*x15 - u2 - 992422053
-167050431*u0^3 + 688082254*u1^3 + 632975430*u2^3 - 853746491*x0 + 582962269*x1 + 316358424*x2 - 702326490*x3 - 898551238*x4 - 52368114*x5 - 942183640*x6 - 913659941*x7 - 99952136*x8 - 1022859214*x9 - 691209880*x10 + 189019033*x11 - 1052571021*x12 - 707721564*x13 - 319546712*x14 - 319546712*x15 - u3 + 212688860
-192747735*u0^3 - 167050431*u1^3 + 688082254*u2^3 + 632975430*u3^3 + 573376230*x0 - 126391743*x1 + 337

### Part2.3: Align the high-degree component space

In [ ]:


name_to_gen = {str(g): g for g in R.gens()}

def _u_num(name):
    return int(name[1:])

U_names = sorted([nm for nm in name_to_gen if nm.startswith("u") and nm[1:].isdigit()],
                 key=_u_num)
U = [name_to_gen[nm] for nm in U_names]

Ec = len(reduced_ideal)
assert len(eqs) == Ec, f"Expected len(eqs)=len(reduced_ideal), got {len(eqs)} and {Ec}"
assert len(U) >= 2*Ec + 1, f"Need at least u0..u{2*Ec}, but only found {len(U)} U variables"

base_U = U[:Ec]                  # u0..u11
fixed_u = U[Ec]                  # u12
target_U = U[Ec+1:Ec+1+Ec]       # u13..u24

PRINT_FULL_RELATIONS = False     

print("Ec =", Ec)
print("base_U   =", [str(u) for u in base_U])
print("fixed_u  =", fixed_u, "= 1")
print("target_U =", [str(u) for u in target_U])


def eliminate_base_cubes(f, red_sub, base_U, *, verbose=False):
    """

    """
    lambdas = {}
    for i in reversed(range(len(base_U))):
        mon = base_U[i]**3
        c_f = f.coefficient(mon)
        if c_f == 0:
            lambdas[i] = R(0)
            continue

        c_g = red_sub[i].coefficient(mon)
        if c_g == 0:
            raise ValueError(f"reduced_ideal[{i}] has zero coefficient on {base_U[i]}^3")

        lam = c_f / c_g
        f = f - lam * red_sub[i]
        lambdas[i] = lam

        if verbose:
            print(f"  eliminate {base_U[i]}^3: lambda =", lam)

    return f, lambdas



subs_fixed = {fixed_u: R(1)}
red_fixed = [f.subs(subs_fixed) for f in reduced_ideal]

linear_relations = []      #
rhs_relations = []         # 
lambda_records = []

for j, target in enumerate(target_U):
    # 
    subs_u = dict(subs_fixed)
    for k in range(j):
        subs_u[target_U[k]] = base_U[k]

    red_sub = [f.subs(subs_u) for f in reduced_ideal]
    f = eqs[j].subs(subs_u)

    print("\n" + "="*80)
    print(f"eqs[{j}]: substitute {fixed_u}=1 and "
          f"{', '.join([str(target_U[k])+'='+str(base_U[k]) for k in range(j)]) if j else 'no previous target U'}")
    print("target =", target)

    f, lambdas = eliminate_base_cubes(f, red_sub, base_U, verbose=False)
    lambda_records.append(lambdas)

    target_coeff = f.coefficient(target)
    print("coeff of", target, "=", target_coeff)

    if target_coeff == 0:
        print("WARNING: no target term found; inspect this relation manually.")
        rhs = None
    else:
        rhs = -(f - target_coeff * target) / target_coeff
        print(f"{target} =", rhs if PRINT_FULL_RELATIONS else "<linear expression; set PRINT_FULL_RELATIONS=True to print>")

    
    leftover_cubes = []
    for i, u in enumerate(base_U):
        cf = f.coefficient(u**3)
        if cf != 0:
            leftover_cubes.append((i, u, cf))
    if leftover_cubes:
        print("WARNING: leftover cubic terms:", leftover_cubes)
    else:
        print("all base cubic terms eliminated")

    if PRINT_FULL_RELATIONS:
        print("final relation f =", f)

    linear_relations.append(f)
    rhs_relations.append(rhs)

print("\n===== summary =====")
print("built", len(linear_relations), "linear relations for", [str(u) for u in target_U])

Ec = 12
base_U   = ['u0', 'u1', 'u2', 'u3', 'u4', 'u5', 'u6', 'u7', 'u8', 'u9', 'u10', 'u11']
fixed_u  = u12 = 1
target_U = ['u13', 'u14', 'u15', 'u16', 'u17', 'u18', 'u19', 'u20', 'u21', 'u22', 'u23', 'u24']

eqs[0]: substitute u12=1 and no previous target U
target = u13
coeff of u13 = -1
u13 = <linear expression; set PRINT_FULL_RELATIONS=True to print>
all base cubic terms eliminated

eqs[1]: substitute u12=1 and u13=u0
target = u14
coeff of u14 = -1
u14 = <linear expression; set PRINT_FULL_RELATIONS=True to print>
all base cubic terms eliminated

eqs[2]: substitute u12=1 and u13=u0, u14=u1
target = u15
coeff of u15 = -1
u15 = <linear expression; set PRINT_FULL_RELATIONS=True to print>
all base cubic terms eliminated

eqs[3]: substitute u12=1 and u13=u0, u14=u1, u15=u2
target = u16
coeff of u16 = -1
u16 = <linear expression; set PRINT_FULL_RELATIONS=True to print>
all base cubic terms eliminated

eqs[4]: substitute u12=1 and u13=u0, u14=u1, u15=u2, u16=u3
target = u17
coeff of u17 = 

### Part2.4: Solve the remainder constraints in the low-degree space and deriving the resulting linear system to obtain affine linear expressions of \(U\) in terms of \(X\).

In [ ]:


closure_relations = []
for j, target in enumerate(target_U):
    g = linear_relations[j].subs({target: base_U[j]})
    closure_relations.append(g)

zero_base_U = {u: R(0) for u in base_U}
A_rows = []
b_entries = []

print("===== extracting linear system in", [str(u) for u in base_U], "=====")
for j, g in enumerate(closure_relations):
    row = [g.coefficient(u) for u in base_U]
    const = g.subs(zero_base_U)

  
    lin_part = sum(row[i] * base_U[i] for i in range(Ec))
    residual = g - lin_part - const
    if residual != 0:
        print(f"WARNING: closure relation {j} is not purely linear in base_U; residual =", residual)

    A_rows.append(row)
    b_entries.append(-const)

A = matrix(R, A_rows)
b = vector(R, b_entries)

print("A size =", A.nrows(), "x", A.ncols())
print("b length =", len(b))

try:
    detA = A.det()
    print("det(A) =", detA)
except Exception as e:
    detA = None
    print("det(A) failed:", e)

try:
    sol = A.solve_right(b)
    solve_parent = R
    print("solve_right over R succeeded")
except Exception as e:
    print("solve_right over R failed; retry over FractionField(R)")
    print("reason:", e)
    K = FractionField(R)
    A_K = matrix(K, A.nrows(), A.ncols(), [K(a) for a in A.list()])
    b_K = vector(K, [K(x) for x in b])
    sol = A_K.solve_right(b_K)
    solve_parent = K
    print("solve_right over FractionField(R) succeeded")

U_expr = {}
for i, u in enumerate(base_U):
    U_expr[i] = sol[i]

U_expr[Ec] = solve_parent(1) if solve_parent is not R else R(1)   # u12 = 1

for j in range(Ec):
    U_expr[Ec + 1 + j] = U_expr[j]   # u13=u0, ..., u24=u11

subs_all_u = {U[i]: U_expr[i] for i in range(2*Ec + 1)}


for i in range(2*Ec + 1):
    globals()[f"u{i}_expr"] = U_expr[i]

print("\n===== solved U affine forms =====")
for i in range(2*Ec + 1):
    print(f"u{i} =", U_expr[i])

print("\nsubs_all_u contains", len(subs_all_u), "substitutions")

===== extracting linear system in ['u0', 'u1', 'u2', 'u3', 'u4', 'u5', 'u6', 'u7', 'u8', 'u9', 'u10', 'u11'] =====
A size = 12 x 12
b length = 12
det(A) = -21254548
solve_right over R succeeded

===== solved U affine forms =====
u0 = (-760261364*x0 + 756411509*x1 - 174466970*x2 + 714291793*x3 - 276602964*x4 - 922766310*x5 + 406375117*x6 + 706232082*x7 + 484964868*x8 + 339435502*x9 - 164260496*x10 - 935017158*x11 - 246561209*x12 + 117127919*x13 - 337985138*x14 - 337985138*x15 - 262347813)/(-594846004)
u1 = (328208719*x0 + 889365841*x1 - 686337099*x2 - 498670980*x3 + 1001125797*x4 - 612766965*x5 - 296990876*x6 - 423488201*x7 + 283431594*x8 + 20756448*x9 + 321890357*x10 + 335705560*x11 - 456438186*x12 + 455507160*x13 + 674846398*x14 + 674846398*x15 + 991738027)/(-594846004)
u2 = (873266492*x0 - 640622851*x1 - 333684570*x2 + 203126543*x3 - 1028819039*x4 + 623877794*x5 - 864381883*x6 - 44346396*x7 - 707675048*x8 + 678325523*x9 - 955858884*x10 + 1035794619*x11 + 394775789*x12 + 959183651*x13

In [ ]:
# ===== substitute solved U expressions back into all reduced_ideal and eqs =====
#####red_final reflects the former 12 constraints and eqs_final reflexts last 12 constraints.

red_final = [f.subs(subs_all_u) for f in reduced_ideal]
eqs_final = [f.subs(subs_all_u) for f in eqs]

PRINT_FINAL_EQUATIONS = True

print("===== reduced_ideal after substituting solved U =====")
for i, f in enumerate(red_final):
    if PRINT_FINAL_EQUATIONS:
        print(f"reduced[{i}] =", f)
    else:
        print(f"reduced[{i}] done")

print("\n===== eqs after substituting solved U =====")
for i, f in enumerate(eqs_final):
    if PRINT_FINAL_EQUATIONS:
        print(f"eqs[{i}] =", f)
    else:
        print(f"eqs[{i}] done")

all_final = list(red_final) + list(eqs_final)
print("\nTotal final equations =", len(all_final))

===== reduced_ideal after substituting solved U =====
reduced[0] = (-352892625*x0^3 - 598671304*x0^2*x1 - 837688754*x0*x1^2 + 1031402714*x1^3 - 927989609*x0^2*x2 - 668827166*x0*x1*x2 - 855312226*x1^2*x2 + 712300306*x0*x2^2 - 535673580*x1*x2^2 + 752618429*x2^3 - 945312399*x0^2*x3 + 411077980*x0*x1*x3 - 91721829*x1^2*x3 + 502012769*x0*x2*x3 + 219713619*x1*x2*x3 + 932309912*x2^2*x3 + 254460231*x0*x3^2 + 93225417*x1*x3^2 + 168729109*x2*x3^2 - 1041690281*x3^3 - 837695683*x0^2*x4 + 108318183*x0*x1*x4 + 33473940*x1^2*x4 + 274069536*x0*x2*x4 - 858145784*x1*x2*x4 - 875523379*x2^2*x4 + 856099010*x0*x3*x4 - 497726513*x1*x3*x4 - 850607192*x2*x3*x4 + 1037311019*x3^2*x4 - 366657312*x0*x4^2 - 700949503*x1*x4^2 + 1055652799*x2*x4^2 - 16909995*x3*x4^2 + 200970357*x4^3 - 75652639*x0^2*x5 + 397577295*x0*x1*x5 - 20550933*x1^2*x5 - 120710925*x0*x2*x5 + 634829990*x1*x2*x5 + 405986725*x2^2*x5 - 289401630*x0*x3*x5 + 713893099*x1*x3*x5 + 456850779*x2*x3*x5 - 269660622*x3^2*x5 + 963881803*x0*x4*x5 + 481382862*x

### Part 3: Verification of the constructed ideal

The third part verifies that the constructed ideal satisfies
the expected degeneration properties.

More precisely, it checks the relevant ideal-membership, reduction, and
dimension conditions predicted by the nonlinear subspace construction in
Section 3.3. This provides a direct computational verification that the
constructed ideal has the intended behavior along the internal rounds.

### Part 3.1: GB-free membership certificate check

This cell verifies the ideal-membership relation without computing a Gröbner
basis.

During the high-degree cancellation step, the coefficients
\(\lambda_{j,i}\) were recorded so that

\[
\mathrm{linear\_relations}[j]
=
\mathrm{eqs}[j]
-
\sum_i \lambda_{j,i}\,\mathrm{reduced\_ideal}[i].
\]

After substituting the solved affine-linear expressions for \(U\) in terms of
\(X\), the remaining linear relation is expected to vanish. Therefore, for each
checked index \(j\), we obtain the explicit certificate

\[
\mathrm{eqs\_final}[j]
=
\sum_i \lambda_{j,i}^{\mathrm{final}}\,
\mathrm{red\_final}[i].
\]

Thus the cell checks that

\[
\mathrm{eqs\_final}[j]\in \langle \mathrm{red\_final}\rangle
\]

by direct polynomial identities, rather than by a Gröbner basis reduction.
The final output confirms that all checked equations are explicitly contained
in the ideal generated by `red_final`.

In [ ]:


CHECK_EQ_INDICES = list(range(Ec))   # 
# CHECK_EQ_INDICES = [0]             # 

PRINT_NONZERO_COEFFS = False
PRINT_BAD_REMAINDER = True


def is_zero_rational(expr):
    """

    """
    if expr == 0:
        return True, expr

    try:
        num = expr.numerator()
        return num == 0, num
    except Exception:
        return expr == 0, expr


bad = []

print("===== GB-free membership certificate check =====")

for j in CHECK_EQ_INDICES:
    # combo = sum_i lambda_final_i * red_final[i]
    combo = eqs_final[j] * 0
    nonzero_coeffs = []

    for i in range(Ec):
        lam = lambda_records[j].get(i, R(0))
        if lam == 0:
            continue

        lam_final = lam.subs(subs_all_u)

        if lam_final != 0:
            nonzero_coeffs.append((i, lam_final))

        combo += lam_final * red_final[i]


    lin_final = linear_relations[j].subs(subs_all_u)


    cert_res = eqs_final[j] - combo - lin_final
    cert_ok, cert_num = is_zero_rational(cert_res)


    lin_ok, lin_num = is_zero_rational(lin_final)

    mem_res = eqs_final[j] - combo
    mem_ok, mem_num = is_zero_rational(mem_res)

    ok = cert_ok and lin_ok and mem_ok

    print(f"\n--- eqs_final[{j}] ---")
    print("certificate identity ok? ", cert_ok)
    print("linear relation -> 0?   ", lin_ok)
    print("in <red_final>?         ", mem_ok)
    print("nonzero coeff count     =", len(nonzero_coeffs))

    if PRINT_NONZERO_COEFFS:
        for i, c in nonzero_coeffs:
            print(f"  coeff of red_final[{i}] =", c)

    if not ok:
        bad.append(j)

        if PRINT_BAD_REMAINDER:
            if not cert_ok:
                print("BAD certificate residual numerator =", cert_num)
            if not lin_ok:
                print("BAD linear relation numerator =", lin_num)
            if not mem_ok:
                print("BAD membership residual numerator =", mem_num)

print("\n===== summary =====")
if bad:
    print("Failed indices:", bad)
else:
    print("All checked eqs_final[j] are explicitly in <red_final>.")

===== GB-free membership certificate check =====

--- eqs_final[0] ---
certificate identity ok?  True
linear relation -> 0?    True
in <red_final>?          True
nonzero coeff count     = 12

--- eqs_final[1] ---
certificate identity ok?  True
linear relation -> 0?    True
in <red_final>?          True
nonzero coeff count     = 12

--- eqs_final[2] ---
certificate identity ok?  True
linear relation -> 0?    True
in <red_final>?          True
nonzero coeff count     = 12

--- eqs_final[3] ---
certificate identity ok?  True
linear relation -> 0?    True
in <red_final>?          True
nonzero coeff count     = 12

--- eqs_final[4] ---
certificate identity ok?  True
linear relation -> 0?    True
in <red_final>?          True
nonzero coeff count     = 12

--- eqs_final[5] ---
certificate identity ok?  True
linear relation -> 0?    True
in <red_final>?          True
nonzero coeff count     = 12

--- eqs_final[6] ---
certificate identity ok?  True
linear relation -> 0?    True
in <red_final>? 

### Part 4: Saving the explicit polynomial path

The fourth part records the explicit polynomial path to external files. 
These saved outputs are included to make the construction
inspectable and to provide the explicit data referred to in the paper.

### Part 4: Reconstructing and exporting the explicit polynomial path

This cell reconstructs the explicit nonlinear state path from \(S_0\) to
\(S_{24}\), using the cyclic relations

\[
u_{12}=1,\qquad u_{13}=u_0,\ldots,u_{24}=u_{11}.
\]

It then runs one additional internal round from \(S_{24}\) and extracts the
first coordinate to derive \(u_{25}\). 

The purpose of this cell is to export the explicit polynomial path to a TeX
file. 

In [ ]:


TEX_FILENAME = "nonlinear_trial_t16X16.tex"   # 你自己改文件名
PRINT_U25 = True
EXPORT_ALL_STATES = True


# ------------------------------------------------------------

# ------------------------------------------------------------
subs_cycle = {U[Ec]: R(1)}
for j in range(Ec):
    subs_cycle[U[Ec + 1 + j]] = U[j]

red_cycle = [f.subs(subs_cycle) for f in reduced_ideal]


def reduce_by_red_cycle(expr):
    """

    """
    f = R(expr).subs(subs_cycle)
    f_red, _ = eliminate_base_cubes(f, red_cycle, base_U, verbose=False)
    return f_red.subs(subs_all_u)


# ------------------------------------------------------------

# ------------------------------------------------------------
def rebuild_state_checkpoints_to_u25():
    """

    """
    rc0_first = R(poseidon.round_constants[0][0])


    state = vector(R, [
        U[0] - rc0_first,
        X[0],
        X[1],
        X[2],
        X[3],
        X[4],
        X[5],
        X[6],
        X[7],
        X[8],
        X[9],
        X[10],
        X[11],
        X[12],
        X[13],
        X[14] + X[15],
    ])

    checkpoints = []
    checkpoints.append(("S_0", "initial state with u0", state))

    round_idx = 0


    for i in range(Ec):
        state = poseidon.rp_round_polynomial(state, round_idx)
        round_idx += 1

        u_idx = i + 1
        state = vector(R, [
            U[u_idx] - R(poseidon.round_constants[u_idx][0])
        ] + list(state[1:]))

        checkpoints.append((f"S_{u_idx}", f"after replacing first coordinate by u{u_idx}", state))


    for i in range(Ec):
        state = poseidon.rp_round_polynomial(state, round_idx)
        round_idx += 1

        u_idx = Ec + 1 + i
        state = vector(R, [
            U[u_idx] - R(poseidon.round_constants[u_idx][0])
        ] + list(state[1:]))

        checkpoints.append((f"S_{u_idx}", f"after replacing first coordinate by u{u_idx}", state))


    state25_raw = poseidon.rp_round_polynomial(state, round_idx)
    checkpoints.append(("S_25", "one extra r_p round from S_24; first coordinate gives u25", state25_raw))

    return checkpoints, round_idx


checkpoints_raw, last_round_idx = rebuild_state_checkpoints_to_u25()

print("last_round_idx used for extra round =", last_round_idx)
print("number of checkpoints =", len(checkpoints_raw))


# ------------------------------------------------------------

# ------------------------------------------------------------
state25_raw = checkpoints_raw[-1][2]
u25_raw_first = state25_raw[0]

u25_expr = reduce_by_red_cycle(u25_raw_first)


if len(poseidon.round_constants) > 25:
    u25_next_sbox_expr = u25_expr + R(poseidon.round_constants[25][0])
else:
    u25_next_sbox_expr = None

if PRINT_U25:
    print("\n===== u25 expression =====")
    print("u25 =", u25_expr)
    if u25_next_sbox_expr is not None:
        print("\nIf using next-S-box-input convention:")
        print("u25_next_sbox = u25 + RC[25][0] =", u25_next_sbox_expr)


# ------------------------------------------------------------

# ------------------------------------------------------------
state_records = []

if EXPORT_ALL_STATES:
    print("\n===== reducing all checkpoint states by red_eqs =====")
    for label, desc, st in checkpoints_raw:
        print("reducing", label, "...")

        st_reduced = vector(
            [reduce_by_red_cycle(coord) for coord in st]
        )

        state_records.append((label, desc, st_reduced))

    print("all states reduced")


# ------------------------------------------------------------

# ------------------------------------------------------------
def tex_vec(v):
    return "\\begin{pmatrix}\n" + " \\\\\n".join(latex(a) for a in v) + "\n\\end{pmatrix}"


def tex_mat(M):
    rows = []
    for i in range(M.nrows()):
        rows.append(" & ".join(latex(M[i, j]) for j in range(M.ncols())))
    return "\\begin{pmatrix}\n" + " \\\\\n".join(rows) + "\n\\end{pmatrix}"


def tex_section(title):
    return "\n\n\\section*{" + title + "}\n"


# ------------------------------------------------------------

# ------------------------------------------------------------
used_rc_count = min(26, len(poseidon.round_constants))

with open(TEX_FILENAME, "w", encoding="utf-8") as fh:
    fh.write("% Auto-generated experiment record\n")
    fh.write("% You can rename/edit this file manually.\n\n")

    fh.write("\\documentclass{article}\n")
    fh.write("\\usepackage{amsmath,amssymb}\n")
    fh.write("\\usepackage[margin=1in]{geometry}\n")
    fh.write("\\allowdisplaybreaks\n")
    fh.write("\\begin{document}\n")

    # Basic parameters
    fh.write(tex_section("Basic parameters"))
    fh.write("\\[\n")
    fh.write(f"p = {latex(specific_p)},\\quad ")
    fh.write(f"t = {latex(poseidon.t)},\\quad ")
    fh.write(f"\\alpha = {latex(poseidon.alpha)},\\quad ")
    fh.write(f"E_c = {latex(Ec)}.\n")
    fh.write("\\]\n")

    fh.write("\\[\n")
    fh.write("constants = " + tex_vec(vector(R, [R(c) for c in constants])) + "\n")
    fh.write("\\]\n")

    # Round constants
    fh.write(tex_section("Round constants"))
    fh.write(f"Only the first {used_rc_count} round constants are recorded here.\n")
    fh.write("\\begin{align*}\n")
    for r in range(used_rc_count):
        rc_vec = vector(R, [R(a) for a in poseidon.round_constants[r]])
        fh.write(f"RC_{{{r}}} &= {tex_vec(rc_vec)}")
        if r != used_rc_count - 1:
            fh.write(" \\\\\n")
        else:
            fh.write("\n")
    fh.write("\\end{align*}\n")

    # MDS matrix
    fh.write(tex_section("Cauchy MDS matrix"))
    M_R = matrix(R, poseidon.mds_matrix.nrows(), poseidon.mds_matrix.ncols(),
                 [R(a) for a in poseidon.mds_matrix.list()])
    fh.write("\\[\nM = " + tex_mat(M_R) + "\n\\]\n")

    # U relations
    fh.write(tex_section("U relations"))
    fh.write("\\begin{align*}\n")

    for i in range(Ec):
        fh.write(f"u_{{{i}}} &= {latex(U_expr[i])} \\\\\n")

    fh.write(f"u_{{{Ec}}} &= 1 \\\\\n")

    for j in range(Ec):
        left_idx = Ec + 1 + j
        right_idx = j
        fh.write(
            f"u_{{{left_idx}}} &= u_{{{right_idx}}}"
            f" = {latex(U_expr[right_idx])} \\\\\n"
        )

    fh.write(f"u_{{25}} &= {latex(u25_expr)}")
    if u25_next_sbox_expr is not None:
        fh.write(" \\\\\n")
        fh.write(f"u_{{25}}^{{\\mathrm{{next}}}} &= {latex(u25_next_sbox_expr)}")
    fh.write("\n\\end{align*}\n")

    # State records
    if EXPORT_ALL_STATES:
        fh.write(tex_section("State records after substituting U"))
        fh.write("Each state is reduced by the triangular red equations and then all $u_i$ are substituted.\n")

        for label, desc, st in state_records:
            fh.write("\n\\subsection*{" + label.replace("_", "\\_") + "}\n")
            fh.write("\\textit{" + desc.replace("_", "\\_") + "}\\\\\n")
            fh.write("\\[\n")
            fh.write(label.replace("_", "\\_") + " = " + tex_vec(st) + "\n")
            fh.write("\\]\n")

    fh.write("\\end{document}\n")

print("\n===== TeX export done =====")
print("written to:", TEX_FILENAME)

last_round_idx used for extra round = 24
number of checkpoints = 26

===== u25 expression =====
u25 = (724301108*x0 + 300126737*x1 - 891557631*x2 + 865453287*x3 + 175498220*x4 + 5165975*x5 - 685038448*x6 - 843434582*x7 + 461063263*x8 + 318764591*x9 + 490602882*x10 - 55708898*x11 + 759590345*x12 - 383188951*x13 - 432207336*x14 - 432207336*x15 + 386478130)/983012520

If using next-S-box-input convention:
u25_next_sbox = u25 + RC[25][0] = (724301108*x0 + 300126737*x1 - 891557631*x2 + 865453287*x3 + 175498220*x4 + 5165975*x5 - 685038448*x6 - 843434582*x7 + 461063263*x8 + 318764591*x9 + 490602882*x10 - 55708898*x11 + 759590345*x12 - 383188951*x13 - 432207336*x14 - 432207336*x15 + 370744068)/983012520

===== reducing all checkpoint states by red_eqs =====
reducing S_0 ...
reducing S_1 ...
reducing S_2 ...
reducing S_3 ...
reducing S_4 ...
reducing S_5 ...
reducing S_6 ...
reducing S_7 ...
reducing S_8 ...
reducing S_9 ...
reducing S_10 ...
reducing S_11 ...
reducing S_12 ...
reducing S_13 ..